<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/16_final_inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB16 — Consolidação, Inventário, Rastreabilidade e Fechamento

## 1. Objetivo

Este notebook consolida a trilha final do pipeline PPCOMP/DM, com foco em inventário, rastreabilidade, validação documental e síntese dos resultados obtidos nos notebooks anteriores.

O NB16 não tem a função de recalcular modelos, redefinir cenários, alterar bases de dados ou abrir novos experimentos. Seu papel é organizar as evidências já produzidas, registrar decisões metodológicas, identificar a origem dos principais números e preparar uma visão final para dissertação, qualificação e defesa.

## 2. Escopo

O escopo do NB16 contempla:

1. inventariar os artefatos produzidos do NB00 ao NB15;
2. registrar caminhos, tipos de arquivo, tamanhos, datas de modificação e hashes SHA-256;
3. organizar uma matriz de rastreabilidade por notebook;
4. consolidar decisões metodológicas relevantes;
5. separar a linha de base do artigo da versão dissertativa ampliada;
6. registrar os deltas entre o artigo submetido e a dissertação;
7. verificar a existência dos artefatos esperados dos NBs 10 a 15;
8. consolidar limitações do artigo e como foram endereçadas na dissertação;
9. explicitar o posicionamento do trabalho em relação a PHM/RUL;
10. gerar uma síntese narrativa em Markdown para apoiar qualificação, dissertação e defesa.

## 3. Regra central de fechamento

Este notebook segue a regra de fechamento definida para o NB16:

- não recalcular métricas;
- não reexecutar modelos;
- não alterar rótulos, limiares, episódios ou escores;
- não escolher novo cenário principal;
- não substituir resultados consolidados dos notebooks anteriores;
- apenas validar, consolidar, documentar e apresentar os resultados finais.

## 4. Decisão metodológica preservada

A decisão metodológica pós-NB12 é preservada neste notebook:

- o cenário `W5_K24_H12_P1_TRAIN_M2S` permanece como referência principal da dissertação;
- o cenário `W5_K24_H12_P1_TRAIN_P95` é tratado como sensibilidade forte e competitiva;
- a diferença observada em favor de `train_p95` não reabre a escolha principal, pois ficou abaixo da margem de 5 pontos percentuais definida como critério de governança metodológica;
- essa margem é interpretada como critério de relevância prática, não como teste estatístico.

## 5. Camadas do pipeline

Para fins de rastreabilidade, o pipeline é organizado em duas camadas principais e uma evidência complementar:

### 5.1 Linha de base do artigo

Inclui os notebooks NB00 a NB09, associados à construção inicial da base, análise exploratória, engenharia de atributos, rotulagem, modelagem preliminar e figuras finais utilizadas como base para o artigo submetido.

### 5.2 Versão dissertativa ampliada

Inclui os notebooks NB10 a NB15, associados ao diagnóstico metodológico, baselines temporais simples e suavizados por EWMA causal, sensibilidade, LSTM complementar, análise de custo, antecipabilidade e framework visual final.

### 5.3 Evidência complementar

Inclui o NB13a, associado ao tuning compacto da LSTM. O NB13a produz evidência tabular/JSON e seu diretório de figuras é intencionalmente vazio.

O NB16 atua como camada de fechamento, consolidando as evidências de ambas as fases e registrando a evidência complementar quando aplicável.

## 6. Entradas principais esperadas

O NB16 utiliza artefatos já gerados em `04-reports`, especialmente:

- resumos JSON dos NBs 00 a 15;
- tabelas CSV dos NBs 08 a 15;
- arquivos Parquet usados como insumo ou evidência;
- figuras finais dos NBs 10 a 15;
- manifesto visual do NB15;
- artefatos de calibração, custo, antecipabilidade, sensibilidade e baselines temporais, incluindo o EWMA causal registrado no NB11;
- diretórios de figuras e relatórios;
- diretório vazio intencional do NB13a, quando presente.

## 7. Entregáveis gerados pelo NB16

Este notebook gera os seguintes artefatos de fechamento:

- `16_artifact_inventory.csv`;
- `16_artifact_manifest_sha256.csv`;
- `16_traceability_matrix.csv`;
- `16_expected_artifacts_check.csv`;
- `16_dissertation_figures.csv`;
- `16_intentional_empty_dirs.csv`;
- `16_key_results_summary.csv`;
- `16_methodological_decision_log.csv`;
- `16_article_dissertation_deltas.csv`;
- `16_limitations_addressing.csv`;
- `16_consistency_checks.csv`;
- `16_phm_rul_positioning.md`;
- `16_narrative_synthesis.md`;
- `16_nb16_summary.json`.

## 8. Critérios de aceitação

O NB16 será considerado adequado se:

1. localizar os artefatos principais dos NBs anteriores;
2. gerar inventário rastreável com hash SHA-256;
3. registrar claramente quais resultados pertencem ao artigo e quais pertencem à dissertação;
4. preservar a decisão `train_m2s` versus `train_p95`;
5. registrar classificação mínima por ciclo DSR (`dsr_cycle`) e citabilidade das figuras (`citable`);
6. gerar tabela Figura da dissertação → arquivo de origem;
7. registrar o diretório vazio intencional do NB13a;
8. consolidar as limitações endereçadas pelos NBs 10 a 16;
9. explicitar que o trabalho se aproxima de PHM/RUL apenas como antecipação de criticidade, e não como estimativa formal de Remaining Useful Life;
10. gerar uma síntese narrativa reaproveitável na dissertação e na defesa.

## 9. Observação metodológica

Como notebook de fechamento, o NB16 não deve ser interpretado como etapa experimental. Ele é uma etapa de governança científica, rastreabilidade e preparação textual, destinada a reduzir inconsistências entre notebooks, artefatos, dissertação, artigo e apresentação final.


In [1]:

# =============================================================================
# NB16 — Consolidação, Inventário, Rastreabilidade e Fechamento
# Pipeline PPCOMP/DM — Dissertação de Mestrado
#
# Regra central:
# - Este notebook NÃO recalcula modelos.
# - Este notebook NÃO altera dados, rótulos, cenários, escores ou limiares.
# - Este notebook apenas consolida, valida, inventaria e documenta os artefatos
#   já produzidos pelos notebooks anteriores.
# =============================================================================

from __future__ import annotations

import os
import re
import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1. Configuração de ambiente
# -----------------------------------------------------------------------------

EXECUTION_TS = datetime.now().isoformat(timespec="seconds")

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive").exists() or not any(Path("/content/drive").iterdir()):
        drive.mount("/content/drive")
except Exception:
    # Execução fora do Colab: seguir usando caminhos locais, se existirem.
    pass

PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Mestrado"),
    Path("/content/drive/My Drive/Mestrado"),
    Path.cwd(),
]

PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if p.exists()), Path.cwd())

REPORTS_PATH_CANDIDATES = [
    PROJECT_ROOT / "04-reports",
    Path("/content/drive/MyDrive/Mestrado/04-reports"),
    Path("/content/drive/My Drive/Mestrado/04-reports"),
    Path.cwd() / "04-reports",
]

REPORTS_PATH = next((p for p in REPORTS_PATH_CANDIDATES if p.exists()), REPORTS_PATH_CANDIDATES[0])
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

NOTEBOOKS_PATH_CANDIDATES = [
    PROJECT_ROOT / "PPCOMP_DM" / "notebooks",
    PROJECT_ROOT / "notebooks",
    Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM/notebooks"),
    Path("/content/drive/My Drive/Mestrado/PPCOMP_DM/notebooks"),
]

NOTEBOOKS_PATH = next((p for p in NOTEBOOKS_PATH_CANDIDATES if p.exists()), None)

MAIN_SCENARIO = "W5_K24_H12_P1_TRAIN_M2S"
STRONG_SENSITIVITY_SCENARIO = "W5_K24_H12_P1_TRAIN_P95"
EWMA_BASELINE_METHOD = "EWMA causal sobre fail_rate, com configuração fixa e interpretável no NB11"

ARTICLE_BASELINE_NBS = {f"{i:02d}" for i in range(0, 10)}
DISSERTATION_EXTENSION_NBS = {f"{i:02d}" for i in range(10, 16)}
COMPLEMENTARY_NBS = {"13a"}
CLOSING_NBS = {"16"}

HASH_MAX_BYTES = 800 * 1024 * 1024  # 800 MB; ajuste se quiser hashear arquivos maiores.

# Artefatos textuais gerados pelo próprio NB16.
# Eles podem ser usados como evidência conceitual antes de existirem fisicamente
# no disco, pois serão materializados na seção de persistência desta mesma execução.
NB16_SELF_GENERATED_EVIDENCE = {
    "16_phm_rul_positioning.md",
    "16_narrative_synthesis.md",
}

print("=" * 90)
print("NB16 — Consolidação, Inventário, Rastreabilidade e Fechamento")
print("=" * 90)
print(f"Execução iniciada em: {EXECUTION_TS}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"REPORTS_PATH : {REPORTS_PATH}")
print(f"NOTEBOOKS_PATH: {NOTEBOOKS_PATH if NOTEBOOKS_PATH else 'não localizado'}")
print(f"Cenário principal preservado     : {MAIN_SCENARIO}")
print(f"Sensibilidade forte preservada   : {STRONG_SENSITIVITY_SCENARIO}")
print("=" * 90)


# -----------------------------------------------------------------------------
# 2. Funções utilitárias
# -----------------------------------------------------------------------------

def sha256_file(path: Path, max_bytes: int = HASH_MAX_BYTES) -> Tuple[Optional[str], str]:
    """Calcula SHA-256 se o arquivo existir e não exceder max_bytes."""
    try:
        if not path.exists() or not path.is_file():
            return None, "missing"
        size = path.stat().st_size
        if size > max_bytes:
            return None, f"skipped_gt_{max_bytes}_bytes"
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest(), "ok"
    except Exception as exc:
        return None, f"error: {type(exc).__name__}: {exc}"


def safe_read_json(path: Path) -> Optional[Any]:
    try:
        if not path.exists():
            return None
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except UnicodeDecodeError:
        try:
            with path.open("r", encoding="latin-1") as f:
                return json.load(f)
        except Exception:
            return None
    except Exception:
        return None


def safe_read_text(path: Path, max_chars: int = 20000) -> Optional[str]:
    try:
        if not path.exists():
            return None
        txt = path.read_text(encoding="utf-8", errors="replace")
        return txt[:max_chars]
    except Exception:
        return None


def safe_read_csv(path: Path, **kwargs) -> Optional[pd.DataFrame]:
    try:
        if not path.exists():
            return None
        return pd.read_csv(path, **kwargs)
    except Exception:
        try:
            return pd.read_csv(path, sep=";", **kwargs)
        except Exception:
            return None


def safe_read_parquet_shape(path: Path) -> Tuple[Optional[int], Optional[int], str]:
    try:
        if not path.exists():
            return None, None, "missing"
        df = pd.read_parquet(path)
        return int(df.shape[0]), int(df.shape[1]), "ok"
    except Exception as exc:
        return None, None, f"not_read: {type(exc).__name__}"


def flatten_json(obj: Any, prefix: str = "") -> Dict[str, Any]:
    """Achata JSONs aninhados para facilitar extração e auditoria."""
    out: Dict[str, Any] = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_prefix = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_json(v, new_prefix))
    elif isinstance(obj, list):
        # Evita explodir listas muito grandes. Registra tamanho e primeiros itens simples.
        out[f"{prefix}.__len__"] = len(obj)
        for idx, item in enumerate(obj[:5]):
            new_prefix = f"{prefix}[{idx}]"
            out.update(flatten_json(item, new_prefix))
    else:
        out[prefix] = obj
    return out


def infer_nb_from_path(path: Path) -> Optional[str]:
    """Infere NB pelo nome do arquivo ou diretório.

    Observação: o NB13a é tratado como evidência complementar própria, e não
    como simples artefato do NB13. Por isso, a detecção de "13a" precisa ocorrer
    antes das regras genéricas de dois dígitos.
    """
    s = str(path).replace("\\", "/")
    s_lower = s.lower()
    name = path.name

    # Caso especial: NB13a (tuning compacto da LSTM).
    if (
        re.search(r"(?:^|/)(?:figures_)?nb13a(?:_|/|$)", s_lower)
        or re.search(r"(?:^|/)13a(?:_|/|$)", s_lower)
        or name.lower().startswith("13a_")
    ):
        return "13a"

    patterns = [
        r"(?:^|/)(?:figures_)?nb(\d{2})(?:_|/|$)",
        r"(?:^|/)(\d{2})_",
        r"(?:^|/)(\d{2})[^\d]",
    ]

    for pat in patterns:
        m = re.search(pat, s, flags=re.IGNORECASE)
        if m:
            nb = m.group(1)
            if nb.isdigit() and 0 <= int(nb) <= 16:
                return nb

    m = re.match(r"^(\d{2})_", name)
    if m:
        nb = m.group(1)
        if nb.isdigit() and 0 <= int(nb) <= 16:
            return nb

    return None


def nb_in_inventory_scope(nb: Optional[str], max_numeric_nb: int = 15) -> bool:
    """Define se um identificador NB deve entrar no inventário principal."""
    if nb in COMPLEMENTARY_NBS:
        return True
    return bool(nb and nb.isdigit() and 0 <= int(nb) <= max_numeric_nb)


def artifact_kind(path: Path) -> str:
    suffix = path.suffix.lower()
    name = path.name.lower()

    if suffix in {".png", ".jpg", ".jpeg", ".pdf", ".svg", ".webp"}:
        return "figure_or_visual"
    if suffix in {".csv", ".tsv", ".xlsx", ".xls"}:
        return "table"
    if suffix in {".json"}:
        return "json_summary"
    if suffix in {".parquet", ".feather"}:
        return "dataframe_snapshot"
    if suffix in {".txt", ".md"}:
        return "text_note"
    if suffix in {".ipynb"}:
        return "notebook"
    if "figures" in name:
        return "figure_directory"
    return "other"


def layer_for_nb(nb: Optional[str]) -> str:
    if nb in COMPLEMENTARY_NBS:
        return "evidencia_complementar"
    if nb in ARTICLE_BASELINE_NBS:
        return "primeiro_ciclo_dsr"
    if nb in DISSERTATION_EXTENSION_NBS:
        return "segundo_ciclo_dsr"
    if nb in CLOSING_NBS:
        return "fechamento_nb16"
    return "fora_do_escopo_nb00_nb16"


def dsr_cycle_for_nb(nb: Optional[str]) -> Optional[str]:
    """Classificação mínima por ciclo, sem criar taxonomia pesada."""
    if nb in COMPLEMENTARY_NBS:
        return "complement"
    if nb in ARTICLE_BASELINE_NBS:
        return "1"
    if nb in DISSERTATION_EXTENSION_NBS or nb in CLOSING_NBS:
        return "2"
    return None

def relpath(path: Path, base: Path = REPORTS_PATH) -> str:
    try:
        return str(path.relative_to(base))
    except Exception:
        return str(path)


def as_scalar(value: Any) -> Any:
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return json.dumps(value, ensure_ascii=False)


def first_existing(*names: str) -> Optional[Path]:
    for name in names:
        p = REPORTS_PATH / name
        if p.exists():
            return p
    return None


def find_value_by_key(flat: Dict[str, Any], key_fragments: Iterable[str]) -> Optional[Any]:
    fragments = [k.lower() for k in key_fragments]
    for k, v in flat.items():
        kl = k.lower()
        if all(f in kl for f in fragments):
            return v
    return None


def add_result(rows: List[Dict[str, Any]], source: str, indicator: str, value: Any,
               interpretation: str = "", scenario: str = "") -> None:
    rows.append({
        "source": source,
        "scenario": scenario,
        "indicator": indicator,
        "value": as_scalar(value),
        "interpretation": interpretation,
    })



# -----------------------------------------------------------------------------
# 2.1 Mapa leve de figuras citáveis da dissertação
# -----------------------------------------------------------------------------

# Esta lista não recalcula nada: apenas declara quais figuras devem ser tratadas
# como citáveis, apêndice, indiretas ou não citáveis no inventário e na tabela
# Figura da dissertação -> arquivo de origem.
#
# Vocabulário fechado de citabilidade:
# - yes: citável no corpo da dissertação;
# - appendix: citável preferencialmente em apêndice;
# - indirect: uso narrativo/indireto, sem citação figura a figura;
# - no: artefato legítimo, mas preterido na escrita atual.
CITABLE_VALUES = {"yes", "appendix", "indirect", "no"}

DISSERTATION_FIGURE_REGISTRY: List[Dict[str, Any]] = [
    {
        "dissertation_ref": "SÍNTESE-01",
        "chapter_section": "Cap. 3",
        "source_notebook": "síntese",
        "filename": "framework_completo.pdf",
        "citable": "yes",
        "dsr_cycle": "synthesis",
        "role": "Figura de síntese: visão arquitetural do framework completo.",
        "notes": "A ser criada fora do pipeline, como artefato da dissertação.",
    },
    {
        "dissertation_ref": "SÍNTESE-02",
        "chapter_section": "Cap. 3",
        "source_notebook": "síntese",
        "filename": "ciclo_dsr.pdf",
        "citable": "yes",
        "dsr_cycle": "synthesis",
        "role": "Figura de síntese: ciclos DSR, relevância, rigor e design.",
        "notes": "A ser criada fora do pipeline, como artefato da dissertação.",
    },
    {
        "dissertation_ref": "SÍNTESE-03",
        "chapter_section": "Cap. 3 / Discussão",
        "source_notebook": "síntese",
        "filename": "evolucao_metodologica.pdf",
        "citable": "yes",
        "dsr_cycle": "synthesis",
        "role": "Figura de síntese: NB00–NB09 -> diagnóstico -> NB10 -> NB11–NB16.",
        "notes": "Anotar 390->562 janelas críticas e 285->388 episódios.",
    },
    {
        "dissertation_ref": "Figura metodologia — estados",
        "chapter_section": "Cap. 3",
        "source_notebook": "15",
        "filename": "15_fig_01_diagrama_estados.pdf",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Estados NORMAL/BEFORE/DURING/AFTER.",
        "notes": "Figura canônica do NB15.",
    },
    {
        "dissertation_ref": "Figura metodologia — limiar train-only",
        "chapter_section": "Cap. 3 / Cap. 4",
        "source_notebook": "10",
        "filename": "fig_03_w5_fail_rate_time_thresholds.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Figura central do segundo ciclo DSR: limiar global vs train-only.",
        "notes": "Fonte vigente: figures_nb10_scenarios.",
    },
    {
        "dissertation_ref": "Figura metodologia — categorias de decisão de cenários",
        "chapter_section": "Cap. 3",
        "source_notebook": "10",
        "filename": "fig_07_scenario_decision_categories.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Rastreia a decisão metodológica de seleção de cenários.",
        "notes": "Fonte vigente: figures_nb10_scenarios.",
    },
    {
        "dissertation_ref": "Figura resultados — vencedores por cenário",
        "chapter_section": "Cap. 4.4",
        "source_notebook": "11",
        "filename": "11_fig01_f1_tscv_winners_by_scenario.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "F1 médio dos vencedores por cenário.",
        "notes": "Fonte vigente: figures_nb11.",
    },
    {
        "dissertation_ref": "Figura resultados — baselines",
        "chapter_section": "Cap. 4.4",
        "source_notebook": "11",
        "filename": "11_fig03_baselines_tscv_f1.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Comparação entre modelo tabular vencedor, regras simples e EWMA causal.",
        "notes": "Fonte vigente: figures_nb11.",
    },
    {
        "dissertation_ref": "Figura resultados — sensibilidade F1",
        "chapter_section": "Cap. 4.6",
        "source_notebook": "12",
        "filename": "12_fig01_f1_sensitivity_tscv.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Sensibilidade de F1 TSCV em 25 cenários.",
        "notes": "Fonte vigente: figures_nb12.",
    },
    {
        "dissertation_ref": "Figura resultados — delta F1",
        "chapter_section": "Cap. 4.6",
        "source_notebook": "12",
        "filename": "12_fig02_delta_f1_vs_nb11.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Delta F1 frente ao NB11; cláusula de revisão não acionada.",
        "notes": "Fonte vigente: figures_nb12.",
    },
    {
        "dissertation_ref": "Figura resultados — LSTM",
        "chapter_section": "Cap. 4.6 / Discussão",
        "source_notebook": "13",
        "filename": "13_fig01_lstm_tscv_f1_by_config.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Comparação complementar LSTM.",
        "notes": "Fonte vigente: figures_nb13_lstm.",
    },
    {
        "dissertation_ref": "Figura resultados — recorte temporal",
        "chapter_section": "Cap. 4.5",
        "source_notebook": "15",
        "filename": "15_fig_03_serie_temporal_recorte_ampliado.pdf",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Recorte ampliado da série temporal com episódios e escores.",
        "notes": "Figura canônica do NB15.",
    },
    {
        "dissertation_ref": "Figura decisão — custo por tau",
        "chapter_section": "Cap. 4.5 / Cap. 5.3",
        "source_notebook": "15",
        "filename": "15_fig_05_custo_tau.pdf",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Curva C(tau) e limiares ótimos por razão de custo.",
        "notes": "Figura consolidada pelo NB15 a partir do NB14.",
    },
    {
        "dissertation_ref": "Figura decisão — classes de ação",
        "chapter_section": "Cap. 4.5 / Cap. 5.3",
        "source_notebook": "15",
        "filename": "15_fig_04_classes_acao.pdf",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Zonas interpretativas de decisão por faixa de tau.",
        "notes": "Figura consolidada pelo NB15.",
    },
    {
        "dissertation_ref": "Figura apoio — lead time por duração",
        "chapter_section": "Cap. 4.5 / Discussão",
        "source_notebook": "14",
        "filename": "14_fig04_lead_time_by_duration.png",
        "citable": "yes",
        "dsr_cycle": "2",
        "role": "Lead time por duração de episódio.",
        "notes": "Usar com ressalva dos episódios avaliáveis.",
    },
    {
        "dissertation_ref": "Apêndice — série temporal principal",
        "chapter_section": "Apêndice",
        "source_notebook": "15",
        "filename": "15_fig_02_serie_temporal_principal.pdf",
        "citable": "appendix",
        "dsr_cycle": "2",
        "role": "Visão geral densa da série temporal.",
        "notes": "Preferencialmente em apêndice.",
    },
    {
        "dissertation_ref": "Apêndice/Discussão — duração e antecipabilidade",
        "chapter_section": "Discussão / Apêndice",
        "source_notebook": "15",
        "filename": "15_fig_06_duracao_antecipabilidade.pdf",
        "citable": "appendix",
        "dsr_cycle": "2",
        "role": "Duração × antecipabilidade.",
        "notes": "Figura preferencial para duração × antecipabilidade.",
    },
    {
        "dissertation_ref": "Apêndice/Diagnóstico — calibração",
        "chapter_section": "Apêndice / Diagnóstico",
        "source_notebook": "15",
        "filename": "15_fig_07_calibracao.pdf",
        "citable": "appendix",
        "dsr_cycle": "2",
        "role": "Diagnóstico de calibração.",
        "notes": "Uso pendente de decisão editorial; calibração não é contribuição central.",
    },
    {
        "dissertation_ref": "Artefato preterido — duração NB14",
        "chapter_section": "Não citar",
        "source_notebook": "14",
        "filename": "14_fig03_duration_anticipability.png",
        "citable": "no",
        "dsr_cycle": "2",
        "role": "Artefato legítimo do NB14, preterido pela figura 15_fig_06.",
        "notes": "Manter no acervo; não usar como figura principal.",
    },
]

CITABILITY_BY_FILENAME = {
    item["filename"]: item["citable"]
    for item in DISSERTATION_FIGURE_REGISTRY
}

FIGURE_ROLE_BY_FILENAME = {
    item["filename"]: item["role"]
    for item in DISSERTATION_FIGURE_REGISTRY
}


def citable_for_artifact(path: Path, nb: Optional[str]) -> str:
    """Marca de citabilidade leve, voltada principalmente a figuras."""
    if artifact_kind(path) != "figure_or_visual":
        return ""
    if path.name in CITABILITY_BY_FILENAME:
        return CITABILITY_BY_FILENAME[path.name]
    if dsr_cycle_for_nb(nb) == "1":
        return "indirect"
    return "no"


def find_artifact_source(filename: str) -> Optional[Path]:
    """Localiza fonte ativa de uma figura sem vasculhar backups."""
    candidates = [
        REPORTS_PATH / "dissertation_figures" / filename,
        PROJECT_ROOT / "05-dissertacao" / "figuras" / filename,
        REPORTS_PATH / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    if REPORTS_PATH.exists():
        matches = [
            p for p in REPORTS_PATH.rglob(filename)
            if "_backup_previous_artifacts" not in p.parts
            and "backup" not in p.parts
            and "_backup" not in p.parts
            and "backups" not in p.parts
        ]
        if matches:
            return sorted(matches, key=lambda x: str(x).lower())[0]

    return None


def build_dissertation_figures_table() -> pd.DataFrame:
    rows = []
    for item in DISSERTATION_FIGURE_REGISTRY:
        source_path = find_artifact_source(item["filename"])
        source_exists = source_path is not None and source_path.exists()
        row = dict(item)
        row.update({
            "source_exists": bool(source_exists),
            "source_relative_path": relpath(source_path) if source_exists else "",
            "source_absolute_path": str(source_path) if source_exists else "",
        })
        rows.append(row)
    return pd.DataFrame(rows)


INTENTIONAL_EMPTY_DIRS = [
    {
        "directory": "figures_nb13a_lstm_tuning",
        "status": "intentional_empty",
        "evidence": "tabular_json_only",
        "dsr_cycle": "complement",
        "notebook": "13a",
        "notes": "O NB13a produz evidência tabular/JSON; não gera figuras por construção.",
    }
]

# -----------------------------------------------------------------------------
# 3. Inventário de artefatos
# -----------------------------------------------------------------------------

EXCLUDE_DIR_PARTS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "backup",
    "backups",
    "_backup",
}

# Por padrão, os backups são excluídos do inventário principal para evitar duplicidade.
# Se precisar auditar backups, altere para True.
INCLUDE_BACKUPS = False

artifact_paths: List[Path] = []

if REPORTS_PATH.exists():
    for p in REPORTS_PATH.rglob("*"):
        if any(part in EXCLUDE_DIR_PARTS for part in p.parts):
            continue
        if not INCLUDE_BACKUPS and "_backup_previous_artifacts" in p.parts:
            continue
        if p.is_file():
            nb = infer_nb_from_path(p)
            if nb_in_inventory_scope(nb, max_numeric_nb=15):
                artifact_paths.append(p)

# Opcionalmente, incluir notebooks do repositório, se localizados.
if NOTEBOOKS_PATH and NOTEBOOKS_PATH.exists():
    for p in NOTEBOOKS_PATH.glob("*.ipynb"):
        nb = infer_nb_from_path(p)
        if nb_in_inventory_scope(nb, max_numeric_nb=15):
            artifact_paths.append(p)

artifact_paths = sorted(set(artifact_paths), key=lambda x: str(x).lower())

inventory_rows: List[Dict[str, Any]] = []
manifest_rows: List[Dict[str, Any]] = []

for p in artifact_paths:
    nb = infer_nb_from_path(p)
    stat = p.stat()
    file_hash, hash_status = sha256_file(p)

    row = {
        "nb": nb,
        "layer": layer_for_nb(nb),
        "dsr_cycle": dsr_cycle_for_nb(nb),
        "citable": citable_for_artifact(p, nb),
        "artifact_kind": artifact_kind(p),
        "filename": p.name,
        "relative_path": relpath(p),
        "absolute_path": str(p),
        "suffix": p.suffix.lower(),
        "size_bytes": int(stat.st_size),
        "size_mb": round(stat.st_size / (1024 * 1024), 4),
        "modified_at": datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
        "sha256": file_hash,
        "sha256_status": hash_status,
    }
    inventory_rows.append(row)

    manifest_rows.append({
        "nb": nb,
        "layer": layer_for_nb(nb),
        "dsr_cycle": dsr_cycle_for_nb(nb),
        "filename": p.name,
        "relative_path": relpath(p),
        "size_bytes": int(stat.st_size),
        "modified_at": datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
        "sha256": file_hash,
        "sha256_status": hash_status,
    })


# Registra diretórios intencionalmente vazios que não apareceriam no inventário de arquivos.
intentional_empty_dir_rows: List[Dict[str, Any]] = []
for empty_info in INTENTIONAL_EMPTY_DIRS:
    d = REPORTS_PATH / empty_info["directory"]
    exists = d.exists() and d.is_dir()
    file_count = len([p for p in d.iterdir() if p.is_file()]) if exists else None
    status = empty_info["status"] if exists and file_count == 0 else ("missing" if not exists else "not_empty")
    intentional_empty_dir_rows.append({
        **empty_info,
        "relative_path": relpath(d) if exists else empty_info["directory"],
        "absolute_path": str(d),
        "exists": bool(exists),
        "file_count": file_count,
        "validated_status": status,
    })

    if exists:
        stat = d.stat()
        inventory_rows.append({
            "nb": empty_info["notebook"],
            "layer": layer_for_nb(empty_info["notebook"]),
            "dsr_cycle": empty_info["dsr_cycle"],
            "citable": "",
            "artifact_kind": "intentional_empty_directory",
            "filename": d.name,
            "relative_path": relpath(d),
            "absolute_path": str(d),
            "suffix": "",
            "size_bytes": 0,
            "size_mb": 0.0,
            "modified_at": datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
            "sha256": None,
            "sha256_status": "not_applicable_directory",
        })

empty_dirs_df = pd.DataFrame(intentional_empty_dir_rows)

inventory_df = pd.DataFrame(inventory_rows)
manifest_df = pd.DataFrame(manifest_rows)
dissertation_figures_df = build_dissertation_figures_table()

if inventory_df.empty:
    print("ATENÇÃO: nenhum artefato NB00-NB15 foi localizado em REPORTS_PATH/NOTEBOOKS_PATH.")
else:
    print(f"Artefatos inventariados: {len(inventory_df)}")
    print(inventory_df.groupby(["layer", "nb"]).size().reset_index(name="n").to_string(index=False))


# -----------------------------------------------------------------------------
# 4. Artefatos esperados e matriz de rastreabilidade
# -----------------------------------------------------------------------------

# Observação de rastreabilidade:
# NB04 e NB06 preservam os artefatos canônicos da trilha do artigo sem sufixo _keep_hour0.

NB_METADATA: Dict[str, Dict[str, Any]] = {
    "00": {
        "title": "AED bruta e exploratória",
        "role": "Caracterizar os dados brutos e apoiar a compreensão inicial da distribuição temporal e operacional.",
        "expected": ["00_aed_summary.json", "00_aed_raw_summary.json"],
    },
    "01": {
        "title": "Ingestão e validação",
        "role": "Registrar leitura inicial, validações de estrutura e consistência de entrada.",
        "expected": ["01_ingest_validate_summary.json"],
    },
    "02": {
        "title": "Limpeza e normalização",
        "role": "Normalizar a base e registrar a decisão sobre preservação ou remoção de registros hour==0.",
        "expected": ["02_clean_normalize_summary_keep_hour0.json"],
    },
    "03": {
        "title": "Agregação em janelas",
        "role": "Construir série temporal agregada em janelas, preservando a granularidade adotada.",
        "expected": ["03_window_5min_base_summary_keep_hour0.json"],
    },
    "04": {
        "title": "Detecção de episódios",
        "role": "Detectar janelas críticas e episódios operacionais a partir da taxa de falhas.",
        "expected": ["04_detect_episodes_summary.json"],
    },
    "05": {
        "title": "Engenharia de atributos",
        "role": "Gerar atributos temporais causais para modelagem supervisionada.",
        "expected": ["05_feature_engineering_summary_keep_hour0.json"],
    },
    "06": {
        "title": "Rotulagem de estados",
        "role": "Construir estados NORMAL, BEFORE, DURING e AFTER, além do alvo supervisionado.",
        "expected": ["06_labeling_states_summary.json"],
    },
    "07": {
        "title": "AED consolidada do pipeline",
        "role": "Documentar a versão consolidada do pipeline e validações exploratórias associadas.",
        "expected": ["07_aed_pipeline_summary.json"],
    },
    "08": {
        "title": "Modelagem supervisionada do artigo",
        "role": "Registrar modelos tabulares iniciais, métricas e importância de atributos.",
        "expected": [
            "08_supervised_modeling_summary.json",
            "08_tscv_summary.csv",
            "08_fixed_split_metrics.csv",
            "08_selected_model_test_predictions.parquet",
        ],
    },
    "09": {
        "title": "Figuras e diagnósticos finais do artigo",
        "role": "Gerar figuras finais, calibração e diagnósticos usados como base visual do artigo.",
        "expected": ["09_final_figures_summary.json", "09_probability_diagnostics.csv"],
    },
    "10": {
        "title": "Diagnóstico metodológico de cenários",
        "role": "Avaliar limiares, granularidade, persistência e promover cenários para NB11.",
        "expected": [
            "10_scenario_thresholds_summary.csv",
            "10_scenario_episode_summary.csv",
            "10_scenario_label_impact.csv",
            "10_scenario_decisions.csv",
            "10_scenario_series_states.parquet",
            "10_scenario_episodes.parquet",
            "10_scenario_advanced_to_nb11.json",
            "10_scenario_diagnostics_summary.json",
        ],
    },
    "11": {
        "title": "Baselines, modelos e lead time",
        "role": "Comparar baselines temporais simples, EWMA causal e modelos supervisionados, com calibração e lead time.",
        "expected": [
            "11_metrics_summary.csv",
            "11_winner_model.json",
            "11_baselines_summary.csv",
            "11_lead_time_summary.csv",
            "11_calibration_summary.csv",
            "11_episode_anticipation.csv",
            "11_scores_raw.parquet",
            "11_scores_calibrated.parquet",
            "11_nb11_summary.json",
        ],
    },
    "12": {
        "title": "Sensibilidade e robustez",
        "role": "Avaliar train_p95, P3, referências históricas e cláusula de revisão metodológica.",
        "expected": [
            "12_sensitivity_summary.csv",
            "12_sensitivity_metrics_tscv.csv",
            "12_sensitivity_metrics_fixed_split.csv",
            "12_sensitivity_descriptive.csv",
            "12_train_p95_review.json",
            "12_nb12_summary.json",
        ],
    },
    "13": {
        "title": "LSTM complementar",
        "role": "Avaliar experimento sequencial defensivo sem deslocar o núcleo tabular interpretável.",
        "expected": [
            "13_lstm_summary.json",
            "13_lstm_feature_columns.json",
            "13_lstm_fixed_results_agg.csv",
            "13_lstm_tscv_results_agg.csv",
            "13_lstm_scores.parquet",
            "13_lstm_conclusion.txt",
        ],
    },
    "13a": {
        "title": "Tuning compacto da LSTM",
        "role": "Registrar varredura compacta de hiperparâmetros da LSTM como evidência complementar, sem gerar figuras.",
        "expected": [
            "13a_lstm_tuning_stage1_results.csv",
            "13a_lstm_tuning_stage1_agg.csv",
            "13a_lstm_tuning_stage2_results.csv",
            "13a_lstm_tuning_stage2_agg.csv",
            "13a_lstm_tuning_conclusion.txt",
            "13a_lstm_tuning_best_config.json",
            "13a_lstm_tuning_fixed_results.csv",
            "13a_lstm_tuning_summary.json",
            "13a_lstm_tuning_fixed_results_agg.csv",
            "13a_lstm_tuning_scores.parquet",
            "figures_nb13a_lstm_tuning",
        ],
    },
    "14": {
        "title": "Antecipabilidade e função de custo",
        "role": "Traduzir escores em decisão operacional, lead time, custo C(tau) e limiares ótimos.",
        "expected": [
            "14_cost_curve.csv",
            "14_scores_selected.parquet",
            "14_optimal_tau_by_cost.csv",
            "14_scenario_summary.csv",
            "14_episode_duration_summary.csv",
            "14_episode_anticipability_by_tau.csv",
            "14_threshold_metrics_by_tau.csv",
            "14_false_alerts_by_tau.csv",
            "14_episode_anticipability.csv",
            "14_conclusion_notes.txt",
            "14_nb14_summary.json",
        ],
    },
    "15": {
        "title": "Figuras finais e framework visual",
        "role": "Consolidar figuras finais, classes de ação, tau1/tau2 e manifesto visual.",
        "expected": [
            "15_visual_framework_summary.json",
            "15_visual_framework_manifest.csv",
            "figures_nb15",
        ],
    },
}

expected_rows: List[Dict[str, Any]] = []
trace_rows: List[Dict[str, Any]] = []

for nb, meta in NB_METADATA.items():
    expected = meta.get("expected", [])
    found_count = 0
    missing = []

    for item in expected:
        p = REPORTS_PATH / item
        exists = p.exists()

        # Se item for diretório, também aceitar existência como diretório.
        if not exists:
            matches = list(REPORTS_PATH.glob(item))
            exists = len(matches) > 0

        if exists:
            found_count += 1
        else:
            missing.append(item)

        expected_rows.append({
            "nb": nb,
            "layer": layer_for_nb(nb),
            "expected_artifact": item,
            "exists": bool(exists),
            "expected_path": str(REPORTS_PATH / item),
        })

    actual_count = int((inventory_df["nb"] == nb).sum()) if not inventory_df.empty else 0

    if len(expected) == 0:
        status = "sem_lista_esperada"
    elif found_count == len(expected):
        status = "ok"
    elif found_count > 0:
        status = "parcial"
    else:
        status = "ausente"

    trace_rows.append({
        "nb": nb,
        "title": meta["title"],
        "layer": layer_for_nb(nb),
        "pipeline_role": meta["role"],
        "expected_artifacts": len(expected),
        "expected_found": found_count,
        "actual_artifacts_in_inventory": actual_count,
        "missing_expected_artifacts": "; ".join(missing),
        "status": status,
    })

expected_df = pd.DataFrame(expected_rows)
traceability_df = pd.DataFrame(trace_rows)

print("\nMatriz de rastreabilidade — resumo:")
print(traceability_df[["nb", "title", "expected_found", "expected_artifacts", "actual_artifacts_in_inventory", "status"]].to_string(index=False))


# -----------------------------------------------------------------------------
# 5. Extração não experimental de resultados-chave
# -----------------------------------------------------------------------------
# Atenção: esta seção apenas lê artefatos já gerados.
# Ela não recalcula modelo, rótulo, episódio, escore ou limiar.

key_results: List[Dict[str, Any]] = []
consistency_rows: List[Dict[str, Any]] = []

def add_check(check: str, status: str, evidence: str = "", severity: str = "info") -> None:
    consistency_rows.append({
        "check": check,
        "status": status,
        "severity": severity,
        "evidence": evidence,
    })

# NB10 — decisões e cenários
p_decisions = REPORTS_PATH / "10_scenario_decisions.csv"
df_decisions = safe_read_csv(p_decisions)
if df_decisions is not None:
    add_check("NB10 decisions file", "ok", str(p_decisions))
    decision_cols = [c for c in df_decisions.columns if "decision" in c.lower() or "category" in c.lower()]
    if decision_cols:
        col = decision_cols[0]
        counts = df_decisions[col].value_counts(dropna=False).to_dict()
        add_result(key_results, "10_scenario_decisions.csv", "scenario_decision_counts", counts,
                   "Distribuição das categorias de decisão do NB10.")
    # Procura cenário principal
    contains_main = df_decisions.astype(str).apply(lambda s: s.str.contains(MAIN_SCENARIO, regex=False, na=False)).any().any()
    add_check(
        "Cenário principal presente nas decisões do NB10",
        "ok" if contains_main else "warning",
        MAIN_SCENARIO if contains_main else "não localizado diretamente no CSV",
        "info" if contains_main else "warning",
    )
else:
    add_check("NB10 decisions file", "missing", str(p_decisions), "warning")

p_adv = REPORTS_PATH / "10_scenario_advanced_to_nb11.json"
adv_json = safe_read_json(p_adv)
if adv_json is not None:
    flat = flatten_json(adv_json)
    add_result(key_results, "10_scenario_advanced_to_nb11.json", "json_top_level_keys",
               list(adv_json.keys()) if isinstance(adv_json, dict) else type(adv_json).__name__,
               "Registro dos cenários promovidos ao NB11.")
    contains_main = MAIN_SCENARIO in json.dumps(adv_json, ensure_ascii=False)
    add_check("Cenário principal promovido/registrado para NB11",
              "ok" if contains_main else "warning",
              MAIN_SCENARIO if contains_main else "não localizado no JSON",
              "info" if contains_main else "warning")
else:
    add_check("NB10 advanced_to_nb11", "missing", str(p_adv), "warning")

p_nb10_summary = REPORTS_PATH / "10_scenario_diagnostics_summary.json"
nb10_summary = safe_read_json(p_nb10_summary)
if nb10_summary is not None:
    flat = flatten_json(nb10_summary)
    for fragments, label in [
        (["leakage"], "leakage_statement"),
        (["main", "scenario"], "main_scenario"),
        (["train", "m2s"], "train_m2s_reference"),
        (["episode"], "episode_count_reference"),
    ]:
        v = find_value_by_key(flat, fragments)
        if v is not None:
            add_result(key_results, "10_scenario_diagnostics_summary.json", label, v)

# NB11 — vencedor, métricas, baselines e calibração
p_winner = REPORTS_PATH / "11_winner_model.json"
winner_json = safe_read_json(p_winner)
if winner_json is not None:
    flat = flatten_json(winner_json)
    add_check("NB11 winner model", "ok", str(p_winner))
    for fragments, label in [
        (["scenario"], "nb11_winner_scenario"),
        (["model"], "nb11_winner_model"),
        (["f1"], "nb11_winner_f1"),
        (["recall"], "nb11_winner_recall"),
        (["precision"], "nb11_winner_precision"),
        (["average", "precision"], "nb11_winner_average_precision"),
        (["roc", "auc"], "nb11_winner_roc_auc"),
        (["brier"], "nb11_brier"),
    ]:
        v = find_value_by_key(flat, fragments)
        if v is not None:
            add_result(key_results, "11_winner_model.json", label, v)
else:
    add_check("NB11 winner model", "missing", str(p_winner), "warning")

for csv_name, source_label in [
    ("11_metrics_summary.csv", "Resumo de métricas NB11"),
    ("11_baselines_summary.csv", "Resumo de baselines NB11"),
    ("11_baselines_all.csv", "Detalhamento ampliado de baselines NB11"),
    ("11_lead_time_summary.csv", "Resumo de lead time NB11"),
    ("11_calibration_summary.csv", "Resumo de calibração NB11"),
    ("11_episode_anticipation.csv", "Antecipação de episódios NB11"),
]:
    p = REPORTS_PATH / csv_name
    df = safe_read_csv(p)
    if df is not None:
        add_check(csv_name, "ok", f"{df.shape[0]} linhas × {df.shape[1]} colunas")
        add_result(key_results, csv_name, "shape", f"{df.shape[0]}x{df.shape[1]}", source_label)
    else:
        severity = "info" if csv_name == "11_baselines_all.csv" else "warning"
        add_check(csv_name, "missing", str(p), severity)

# NB11 — registro específico do baseline EWMA causal, quando presente nos artefatos de baseline.
ewma_detected = False
ewma_sources = []
for csv_name in ["11_baselines_summary.csv", "11_baselines_all.csv"]:
    p = REPORTS_PATH / csv_name
    df = safe_read_csv(p)
    if df is None:
        continue
    content = " ".join(df.astype(str).fillna("").to_numpy().ravel()).lower()
    if "ewma" in content:
        ewma_detected = True
        ewma_sources.append(csv_name)
        add_result(
            key_results,
            csv_name,
            "ewma_baseline_registered",
            True,
            "Baseline temporal suavizado por EWMA causal registrado no NB11.",
        )

add_check(
    "NB11 EWMA causal baseline",
    "ok" if ewma_detected else "not_found",
    "; ".join(ewma_sources) if ewma_sources else "EWMA não localizado nos CSVs de baseline disponíveis",
    "info" if ewma_detected else "warning",
)

# NB12 — revisão train_p95
p_p95_review = REPORTS_PATH / "12_train_p95_review.json"
p95_review = safe_read_json(p_p95_review)
if p95_review is not None:
    flat = flatten_json(p95_review)
    add_check("NB12 train_p95 review", "ok", str(p_p95_review))
    add_result(key_results, "12_train_p95_review.json", "review_summary_keys",
               list(p95_review.keys()) if isinstance(p95_review, dict) else type(p95_review).__name__,
               "Revisão metodológica pós-NB12.")
    for fragments, label in [
        (["delta"], "train_p95_delta_vs_train_m2s"),
        (["trigger"], "review_trigger"),
        (["reopen"], "reopen_main_scenario"),
        (["main", "scenario"], "main_scenario_after_nb12"),
    ]:
        v = find_value_by_key(flat, fragments)
        if v is not None:
            add_result(key_results, "12_train_p95_review.json", label, v)
    contains_main = MAIN_SCENARIO in json.dumps(p95_review, ensure_ascii=False)
    contains_p95 = STRONG_SENSITIVITY_SCENARIO in json.dumps(p95_review, ensure_ascii=False)
    add_check("NB12 registra train_m2s e train_p95",
              "ok" if contains_main and contains_p95 else "warning",
              f"main={contains_main}; p95={contains_p95}",
              "info" if contains_main and contains_p95 else "warning")
else:
    add_check("NB12 train_p95 review", "missing", str(p_p95_review), "warning")

# NB13 — LSTM complementar
p_lstm = REPORTS_PATH / "13_lstm_summary.json"
lstm_json = safe_read_json(p_lstm)
if lstm_json is not None:
    flat = flatten_json(lstm_json)
    add_check("NB13 LSTM summary", "ok", str(p_lstm))
    for fragments, label in [
        (["f1"], "nb13_lstm_f1"),
        (["scenario"], "nb13_scenario"),
        (["config"], "nb13_config"),
        (["supersede"], "nb13_supersedes_nb11"),
        (["replace"], "nb13_replaces_nb11_scores"),
    ]:
        v = find_value_by_key(flat, fragments)
        if v is not None:
            add_result(key_results, "13_lstm_summary.json", label, v)
else:
    add_check("NB13 LSTM summary", "missing", str(p_lstm), "warning")

p_lstm_conclusion = REPORTS_PATH / "13_lstm_conclusion.txt"
lstm_conclusion = safe_read_text(p_lstm_conclusion, max_chars=4000)
if lstm_conclusion:
    add_result(key_results, "13_lstm_conclusion.txt", "lstm_conclusion_excerpt",
               lstm_conclusion[:1000].replace("\n", " "),
               "Conclusão textual do experimento complementar LSTM.")
    add_check("NB13 LSTM conclusion", "ok", str(p_lstm_conclusion))

# NB14 — custo e antecipabilidade
p_tau = REPORTS_PATH / "14_optimal_tau_by_cost.csv"
df_tau = safe_read_csv(p_tau)
if df_tau is not None:
    add_check("NB14 optimal tau by cost", "ok", f"{df_tau.shape[0]} linhas × {df_tau.shape[1]} colunas")
    add_result(key_results, "14_optimal_tau_by_cost.csv", "shape", f"{df_tau.shape[0]}x{df_tau.shape[1]}",
               "Tabela de tau ótimo por razão de custo.")
    # Registra uma versão compacta das primeiras linhas, sem recalcular nada.
    add_result(key_results, "14_optimal_tau_by_cost.csv", "head_records",
               df_tau.head(10).to_dict(orient="records"),
               "Amostra das políticas de limiar já calculadas pelo NB14.")
else:
    add_check("NB14 optimal tau by cost", "missing", str(p_tau), "warning")

for csv_name, source_label in [
    ("14_scenario_summary.csv", "Resumo de cenário NB14"),
    ("14_episode_duration_summary.csv", "Duração e antecipabilidade por faixa"),
    ("14_episode_anticipability.csv", "Antecipabilidade por episódio"),
    ("14_episode_anticipability_by_tau.csv", "Antecipabilidade por tau"),
    ("14_threshold_metrics_by_tau.csv", "Métricas por tau"),
    ("14_false_alerts_by_tau.csv", "Falsos alertas por tau"),
    ("14_cost_curve.csv", "Curva custo × tau"),
]:
    p = REPORTS_PATH / csv_name
    df = safe_read_csv(p)
    if df is not None:
        add_check(csv_name, "ok", f"{df.shape[0]} linhas × {df.shape[1]} colunas")
        add_result(key_results, csv_name, "shape", f"{df.shape[0]}x{df.shape[1]}", source_label)
    else:
        add_check(csv_name, "missing", str(p), "warning")

p_nb14_summary = REPORTS_PATH / "14_nb14_summary.json"
nb14_json = safe_read_json(p_nb14_summary)
if nb14_json is not None:
    flat = flatten_json(nb14_json)
    add_check("NB14 summary json", "ok", str(p_nb14_summary))
    for fragments, label in [
        (["score"], "nb14_score_source"),
        (["calibrated"], "nb14_calibration_interpretation"),
        (["main", "scenario"], "nb14_main_scenario"),
    ]:
        v = find_value_by_key(flat, fragments)
        if v is not None:
            add_result(key_results, "14_nb14_summary.json", label, v)

# NB15 — framework visual
p_nb15_summary = REPORTS_PATH / "15_visual_framework_summary.json"
nb15_json = safe_read_json(p_nb15_summary)
if nb15_json is not None:
    flat = flatten_json(nb15_json)
    add_check("NB15 visual framework summary", "ok", str(p_nb15_summary))
    for fragments, label in [
        (["tau1"], "nb15_tau1"),
        (["tau2"], "nb15_tau2"),
        (["main", "scenario"], "nb15_main_scenario"),
        (["figure"], "nb15_figures"),
    ]:
        v = find_value_by_key(flat, fragments)
        if v is not None:
            add_result(key_results, "15_visual_framework_summary.json", label, v)
else:
    add_check("NB15 visual framework summary", "missing", str(p_nb15_summary), "warning")

p_nb15_manifest = REPORTS_PATH / "15_visual_framework_manifest.csv"
df_visual_manifest = safe_read_csv(p_nb15_manifest)
if df_visual_manifest is not None:
    add_check("NB15 visual framework manifest", "ok",
              f"{df_visual_manifest.shape[0]} linhas × {df_visual_manifest.shape[1]} colunas")
    add_result(key_results, "15_visual_framework_manifest.csv", "shape",
               f"{df_visual_manifest.shape[0]}x{df_visual_manifest.shape[1]}",
               "Manifesto das figuras finais do NB15.")

# Parquets principais — registrar apenas shape quando possível.
for parquet_name in [
    "10_scenario_series_states.parquet",
    "10_scenario_episodes.parquet",
    "11_scores_raw.parquet",
    "11_scores_calibrated.parquet",
    "12_scores_sensitivity_fixed.parquet",
    "13_lstm_scores.parquet",
    "14_scores_selected.parquet",
]:
    p = REPORTS_PATH / parquet_name
    rows, cols, status = safe_read_parquet_shape(p)
    add_check(f"Parquet shape: {parquet_name}",
              "ok" if status == "ok" else ("missing" if status == "missing" else "warning"),
              f"rows={rows}; cols={cols}; status={status}",
              "info" if status == "ok" else "warning")
    if status == "ok":
        add_result(key_results, parquet_name, "shape", f"{rows}x{cols}", "Dimensão do artefato Parquet já gerado.")


key_results_df = pd.DataFrame(key_results)
consistency_df = pd.DataFrame(consistency_rows)

print("\nChecagens de consistência — resumo:")
if not consistency_df.empty:
    print(consistency_df.groupby(["status", "severity"]).size().reset_index(name="n").to_string(index=False))
else:
    print("Nenhuma checagem registrada.")

print("\nResultados-chave extraídos de artefatos existentes:")
if not key_results_df.empty:
    print(key_results_df.head(20).to_string(index=False))
else:
    print("Nenhum resultado-chave extraído.")


# -----------------------------------------------------------------------------
# 6. Deltas artigo → dissertação, limitações e decisões metodológicas
# -----------------------------------------------------------------------------

article_deltas_df = pd.DataFrame([
    {
        "axis": "Limiar de criticidade",
        "article_baseline": "Uso de limiar global/referência histórica na formulação inicial.",
        "dissertation_extension": "NB10 introduz e consolida limiares train-only, preservando global_m2s apenas como referência histórica.",
        "evidence_artifacts": "10_scenario_thresholds_summary.csv; 10_scenario_diagnostics_summary.json; 10_scenario_decisions.csv",
        "interpretation": "Reduz risco de vazamento temporal e fortalece a causalidade metodológica.",
    },
    {
        "axis": "Baselines temporais",
        "article_baseline": "Modelos supervisionados comparados sem um conjunto robusto de regras temporais causais.",
        "dissertation_extension": "NB11 compara classe majoritária, regras temporais simples, limiar rebaixado, z-score causal, EWMA causal e modelos tabulares.",
        "evidence_artifacts": "11_baselines_summary.csv; 11_baselines_all.csv; 11_metrics_summary.csv",
        "interpretation": "Permite demonstrar valor agregado dos modelos frente a regras operacionais simples e a um baseline suavizado causal.",
    },
    {
        "axis": "Robustez e sensibilidade",
        "article_baseline": "Sensibilidade metodológica ainda limitada na formulação inicial.",
        "dissertation_extension": "NB12 avalia train_p95, P3, referências históricas e p99 descritivo sem reabrir grid amplo.",
        "evidence_artifacts": "12_sensitivity_summary.csv; 12_train_p95_review.json",
        "interpretation": "Preserva governança metodológica: train_m2s principal e train_p95 sensibilidade forte.",
    },
    {
        "axis": "Modelagem sequencial",
        "article_baseline": "LSTM não era núcleo experimental do artigo.",
        "dissertation_extension": "NB13 avalia LSTM como experimento complementar defensivo, sem deslocar o núcleo interpretável.",
        "evidence_artifacts": "13_lstm_summary.json; 13_lstm_conclusion.txt",
        "interpretation": "Antecipa questionamentos de banca/revisores sem transformar rede neural no eixo central.",
    },
    {
        "axis": "Decisão sob incerteza",
        "article_baseline": "Função de custo C(tau) ainda incompleta ou não operacionalizada.",
        "dissertation_extension": "NB14 operacionaliza custo por tau, razões cFP:cFN, antecipabilidade, falsos alertas e lead time.",
        "evidence_artifacts": "14_cost_curve.csv; 14_optimal_tau_by_cost.csv; 14_threshold_metrics_by_tau.csv",
        "interpretation": "Conecta o problema preditivo à tomada de decisão operacional.",
    },
    {
        "axis": "Framework visual",
        "article_baseline": "Figuras iniciais voltadas ao artigo.",
        "dissertation_extension": "NB15 consolida diagrama de estados, séries temporais, classes de ação, calibração e curva custo×tau.",
        "evidence_artifacts": "15_visual_framework_summary.json; 15_visual_framework_manifest.csv; figures_nb15",
        "interpretation": "Apoia dissertação, qualificação e defesa com figuras rastreáveis.",
    },
    {
        "axis": "PHM/RUL",
        "article_baseline": "Conexão bibliográfica com PHM/RUL pouco explicitada.",
        "dissertation_extension": "NB16 posiciona o trabalho como antecipação de criticidade, não como estimativa formal de RUL.",
        "evidence_artifacts": "16_phm_rul_positioning.md; 16_narrative_synthesis.md",
        "interpretation": "Resolve parcialmente a lacuna conceitual e delimita trabalhos futuros.",
    },
])

limitations_df = pd.DataFrame([
    {
        "article_limitation": "Ausência de baseline temporal robusto.",
        "addressed_by": "NB11",
        "expected_resolution": "Comparação com baselines temporais simples, EWMA causal e modelos supervisionados sob protocolo temporal.",
        "evidence_artifacts": "11_baselines_summary.csv; 11_metrics_summary.csv; 11_metrics_tscv.csv",
        "status": "resolvida_se_artefatos_presentes",
    },
    {
        "article_limitation": "Limiar global com risco de vazamento temporal.",
        "addressed_by": "NB10",
        "expected_resolution": "Introdução de limiares train-only e separação de global_m2s como referência histórica.",
        "evidence_artifacts": "10_scenario_thresholds_summary.csv; 10_scenario_diagnostics_summary.json",
        "status": "resolvida_se_artefatos_presentes",
    },
    {
        "article_limitation": "Função C(tau) incompleta.",
        "addressed_by": "NB14",
        "expected_resolution": "Análise paramétrica de custo e identificação de tau ótimo por razão cFP:cFN.",
        "evidence_artifacts": "14_cost_curve.csv; 14_optimal_tau_by_cost.csv",
        "status": "resolvida_se_artefatos_presentes",
    },
    {
        "article_limitation": "Ausência de conexão explícita com PHM/RUL.",
        "addressed_by": "NB16",
        "expected_resolution": "Posicionamento explícito como antecipação de criticidade, sem reivindicar RUL formal.",
        "evidence_artifacts": "16_phm_rul_positioning.md",
        "status": "parcialmente_resolvida_por_delimitacao_conceitual",
    },
])

# Atualiza status das limitações com base na existência real dos artefatos.
def artifacts_exist_from_string(s: str) -> str:
    """Verifica existência de evidências já materializadas ou geradas pelo próprio NB16.

    Observação importante:
    alguns artefatos textuais do fechamento, como 16_phm_rul_positioning.md,
    são produzidos neste próprio notebook. Portanto, quando aparecem como evidência
    de limitações/decisões do NB16, devem ser tratados como evidências presentes
    desde que estejam registrados em NB16_SELF_GENERATED_EVIDENCE. Essa regra evita
    falso negativo causado pela ordem de execução: checar a evidência antes de o
    arquivo textual ser gravado no disco.
    """
    parts = [x.strip() for x in re.split(r";|,", s) if x.strip()]
    found, missing = [], []

    for item in parts:
        candidate = REPORTS_PATH / item

        if candidate.exists():
            found.append(item)
            continue

        # Aceita diretório ou glob simples.
        matches = list(REPORTS_PATH.glob(item))
        if matches:
            found.append(item)
            continue

        # Evita falso negativo para evidências textuais geradas pelo próprio NB16.
        if item in NB16_SELF_GENERATED_EVIDENCE:
            found.append(f"{item} [gerado_pelo_NB16]")
            continue

        missing.append(item)

    if not parts:
        return "sem_artefato_definido"
    if not missing:
        return "evidencias_presentes"
    if found:
        return "evidencias_parciais"
    return "evidencias_ausentes"

limitations_df["evidence_status"] = limitations_df["evidence_artifacts"].apply(artifacts_exist_from_string)

decision_log_df = pd.DataFrame([
    {
        "decision_id": "D01",
        "decision": "Preservar train_m2s como cenário principal da dissertação.",
        "basis": "Regra de governança pós-NB12: train_p95 só reabriria a escolha principal se superasse train_m2s por mais de 5 pontos percentuais em F1 médio TSCV.",
        "final_position": f"{MAIN_SCENARIO} permanece como referência principal.",
        "evidence_artifacts": "12_train_p95_review.json; 12_nb12_summary.json; 12_sensitivity_summary.csv",
    },
    {
        "decision_id": "D02",
        "decision": "Tratar train_p95 como sensibilidade forte e competitiva.",
        "basis": "Cenário demonstrou desempenho relevante, mas sem acionar a cláusula formal de revisão.",
        "final_position": f"{STRONG_SENSITIVITY_SCENARIO} deve aparecer como robustez/sensibilidade, não como substituto automático.",
        "evidence_artifacts": "12_train_p95_review.json; 12_sensitivity_metrics_tscv.csv",
    },
    {
        "decision_id": "D03",
        "decision": "Manter LSTM como experimento complementar defensivo.",
        "basis": "NB13 testa arquitetura sequencial clássica, mas não desloca o núcleo causal, tabular e interpretável da dissertação.",
        "final_position": "Usar como evidência complementar, especialmente para discussão de limites e trabalhos futuros.",
        "evidence_artifacts": "13_lstm_summary.json; 13_lstm_conclusion.txt",
    },
    {
        "decision_id": "D04",
        "decision": "Interpretar tau como limiar sobre escore calibrado ou escore relativo, conforme conclusão de calibração.",
        "basis": "NB14 consome escores do NB11 ou, condicionalmente, do NB13, sem recalibrar no NB16.",
        "final_position": "A curva C(tau) deve ser descrita como decisão sob incerteza e não como garantia probabilística absoluta.",
        "evidence_artifacts": "11_calibration_summary.csv; 14_nb14_summary.json; 14_cost_curve.csv",
    },
    {
        "decision_id": "D05",
        "decision": "Tratar tau1/tau2 como limiares rastreáveis derivados do NB14 e visualizados no NB15.",
        "basis": "NB15 deve extrair tau1/tau2 prioritariamente dos resultados de custo do cenário principal.",
        "final_position": "Classes de ação são recurso interpretativo para defesa e dissertação.",
        "evidence_artifacts": "14_optimal_tau_by_cost.csv; 15_visual_framework_summary.json",
    },
    {
        "decision_id": "D06",
        "decision": "Não reivindicar RUL formal.",
        "basis": "O pipeline antecipa criticidade em janelas futuras, mas não estima vida útil remanescente de componente físico.",
        "final_position": "Posicionar como antecipação de criticidade e apoio à decisão sob incerteza; PHM/RUL fica como relação conceitual e trabalho futuro.",
        "evidence_artifacts": "16_phm_rul_positioning.md",
    },
    {
        "decision_id": "D07",
        "decision": "Registrar o EWMA causal como baseline temporal suavizado comparativo.",
        "basis": "O EWMA amplia a comparação temporal do NB11 sem substituir o vencedor supervisionado nem a política decisória do NB14.",
        "final_position": "Usar o EWMA como referência intermediária entre regras instantâneas simples e modelos sequenciais não triviais.",
        "evidence_artifacts": "11_baselines_summary.csv; 11_metrics_summary.csv",
    },
])

decision_log_df["evidence_status"] = decision_log_df["evidence_artifacts"].apply(artifacts_exist_from_string)


# -----------------------------------------------------------------------------
# 7. Posicionamento PHM/RUL e síntese narrativa
# -----------------------------------------------------------------------------

def df_to_markdown_safe(df: pd.DataFrame, max_rows: int = 20) -> str:
    if df is None or df.empty:
        return "_Sem registros disponíveis._"
    try:
        return df.head(max_rows).to_markdown(index=False)
    except Exception:
        return df.head(max_rows).to_string(index=False)


trace_summary_md = df_to_markdown_safe(
    traceability_df[["nb", "title", "layer", "expected_found", "expected_artifacts", "status"]],
    max_rows=20,
)

limitations_md = df_to_markdown_safe(
    limitations_df[["article_limitation", "addressed_by", "expected_resolution", "evidence_status"]],
    max_rows=10,
)

decision_md = df_to_markdown_safe(
    decision_log_df[["decision_id", "decision", "final_position", "evidence_status"]],
    max_rows=10,
)

delta_md = df_to_markdown_safe(
    article_deltas_df[["axis", "article_baseline", "dissertation_extension", "interpretation"]],
    max_rows=10,
)

def truncate_text_for_md(value: Any, max_chars: int = 350) -> str:
    """Reduz valores muito longos para manter a síntese em tamanho utilizável."""
    s = str(value)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + " [...]"


def compact_key_results_for_narrative(df: pd.DataFrame, max_rows: int = 24) -> pd.DataFrame:
    """Prepara uma versão enxuta dos resultados-chave para a síntese narrativa.

    O CSV 16_key_results_summary.csv continua preservando os valores completos.
    A compactação afeta apenas o texto 16_narrative_synthesis.md, evitando que
    listas extensas, dicionários e head_records transformem a síntese em um
    documento excessivamente longo.
    """
    if df is None or df.empty:
        return pd.DataFrame()

    compact = df.copy()
    if "indicator" in compact.columns:
        compact = compact[~compact["indicator"].astype(str).isin({"head_records"})]

    keep_cols = [c for c in ["source", "scenario", "indicator", "value"] if c in compact.columns]
    compact = compact[keep_cols].head(max_rows).copy()

    if "value" in compact.columns:
        compact["value"] = compact["value"].apply(truncate_text_for_md)

    return compact


key_results_narrative_df = compact_key_results_for_narrative(key_results_df, max_rows=24)
key_results_md = df_to_markdown_safe(key_results_narrative_df, max_rows=24)

phm_rul_positioning_md = f"""# Posicionamento PHM/RUL — NB16

## Síntese

O pipeline PPCOMP/DM deve ser posicionado como uma abordagem de antecipação de criticidade operacional e apoio à decisão sob incerteza em infraestrutura de TI. Embora exista proximidade conceitual com a literatura de PHM (*Prognostics and Health Management*) e RUL (*Remaining Useful Life*), o trabalho não estima formalmente a vida útil remanescente de um componente físico ou lógico específico.

## Delimitação conceitual

A formulação adotada antecipa a ocorrência de janelas críticas futuras a partir de atributos temporais causais, episódios operacionais e escores de risco. O alvo supervisionado indica se existe episódio `DURING` no horizonte futuro `H`, e não quantos ciclos, horas ou unidades de uso restam até a falha definitiva de um ativo.

Portanto, a contribuição deve ser descrita como:

- antecipação de criticidade;
- identificação de janelas de risco;
- apoio à decisão sob incerteza;
- análise de custo associada a falso positivo e falso negativo;
- priorização operacional baseada em escores temporais.

Não deve ser descrita como:

- estimativa formal de RUL;
- prognóstico de vida útil remanescente;
- modelo de degradação física de componentes;
- predição de tempo exato até falha terminal.

## Relação com PHM

A relação com PHM é metodológica e conceitual. O trabalho compartilha com PHM a preocupação com detecção antecipada, prevenção de falhas, uso de séries temporais e suporte à tomada de decisão. Entretanto, a infraestrutura analisada é tratada por eventos agregados e episódios críticos, não por curvas explícitas de degradação de componentes.

## Relação com RUL

A relação com RUL deve ser apresentada como oportunidade futura. Para transformar esta pesquisa em uma formulação de RUL, seria necessário definir unidades de ativo, trajetórias de degradação, evento terminal, censura, horizonte contínuo até falha e métricas adequadas de erro temporal. Esses elementos não fazem parte do escopo atual.

## Formulação recomendada para a dissertação

Este trabalho se aproxima da literatura de PHM por tratar antecipação de criticidade e prevenção operacional em séries temporais de infraestrutura de TI. No entanto, a formulação proposta não estima formalmente *Remaining Useful Life*. O objetivo é prever a ocorrência de janelas críticas futuras dentro de um horizonte operacional e apoiar decisões sob incerteza por meio de limiares, custos relativos e classes de ação. A extensão para modelos formais de RUL é reconhecida como oportunidade de pesquisa futura.

## Decisão preservada no NB16

O NB16 não altera a decisão metodológica dos notebooks anteriores. O cenário principal permanece `{MAIN_SCENARIO}` e `{STRONG_SENSITIVITY_SCENARIO}` permanece como sensibilidade forte e competitiva.
"""

narrative_synthesis_md = f"""# Síntese narrativa — NB16

_Gerado automaticamente pelo NB16 em {EXECUTION_TS}._

## 1. Trajetória metodológica

A pesquisa evoluiu de uma formulação inicial orientada à construção de uma série temporal de falhas em infraestrutura de TI para uma estrutura mais ampla de antecipação de criticidade e apoio à decisão sob incerteza. A primeira fase do pipeline, representada pelos notebooks NB00 a NB09, consolidou a ingestão, limpeza, agregação temporal, detecção de episódios, engenharia de atributos, rotulagem dos estados operacionais e modelagem supervisionada inicial.

A segunda fase, representada pelos notebooks NB10 a NB15, ampliou a robustez científica da dissertação. O NB10 revisou decisões metodológicas de limiar, granularidade, persistência e horizonte. O NB11 introduziu baselines temporais simples, EWMA causal, modelos supervisionados comparáveis, calibração e lead time. O NB12 avaliou sensibilidade e preservou a decisão metodológica principal. O NB13 testou LSTM como experimento complementar defensivo. O NB14 conectou os escores preditivos à antecipabilidade e à função de custo. O NB15 consolidou as figuras finais e o framework visual.

O NB16 fecha essa trajetória sem recalcular resultados. Sua finalidade é registrar, consolidar e tornar rastreável o conjunto de artefatos produzidos.

## 2. Decisão central preservada

A decisão metodológica final preserva `{MAIN_SCENARIO}` como cenário principal da dissertação. O cenário `{STRONG_SENSITIVITY_SCENARIO}` é reconhecido como sensibilidade forte e competitiva, mas não substitui automaticamente o cenário principal.

Essa decisão decorre da regra de governança metodológica definida após o NB12. A troca do cenário principal somente seria reaberta se `train_p95` superasse `train_m2s` por margem superior a 5 pontos percentuais em F1 médio no TimeSeriesSplit. Como essa condição não foi atendida, a dissertação mantém `train_m2s` como eixo principal e apresenta `train_p95` como evidência de robustez.

A margem de 5 pontos percentuais deve ser interpretada como critério de relevância prática e estabilidade metodológica, não como teste estatístico.

## 3. Achados consolidados

Os artefatos consolidados indicam que o pipeline passou a contemplar quatro dimensões fundamentais: causalidade temporal, robustez metodológica, comparação com baselines e interpretação operacional.

A causalidade temporal foi reforçada pela substituição de decisões baseadas em limiares globais por limiares estimados apenas no treino. A robustez foi ampliada pela análise de cenários alternativos e pela preservação explícita de `train_p95` como sensibilidade. A comparação com baselines foi fortalecida no NB11, permitindo avaliar se os modelos supervisionados agregam valor em relação a regras temporais simples e ao EWMA causal como baseline suavizado. A interpretação operacional foi aprofundada no NB14, com custo por limiar, falsos alertas, falsos negativos e antecipabilidade.

## 4. Linha de base do artigo versus dissertação

A linha de base do artigo corresponde aos notebooks NB00 a NB09. Essa fase estabeleceu a viabilidade do problema, a construção da série temporal, a representação de estados e os primeiros resultados supervisionados.

A dissertação amplia essa base com os notebooks NB10 a NB15. A ampliação não invalida a linha de base do artigo; ela a torna mais defensável. Em particular, a dissertação passa a responder críticas potenciais sobre vazamento temporal, ausência de baseline robusto, falta de análise de sensibilidade, necessidade de conexão com decisão operacional e ausência de posicionamento frente a PHM/RUL.

## 5. Limitações endereçadas

{limitations_md}

## 6. Deltas artigo → dissertação

{delta_md}

## 7. Decisões metodológicas finais

{decision_md}

## 8. Rastreabilidade dos notebooks

{trace_summary_md}

## 9. Resultados-chave extraídos dos artefatos

A tabela abaixo lista uma versão compacta dos indicadores lidos diretamente dos artefatos existentes. Ela não representa recálculo experimental; é apenas uma consolidação de evidências. A versão completa permanece preservada em `16_key_results_summary.csv`.

{key_results_md}

## 10. Posicionamento em relação a PHM/RUL

O trabalho deve ser apresentado como antecipação de criticidade e apoio à decisão sob incerteza, não como estimativa formal de *Remaining Useful Life*. A relação com PHM/RUL é relevante para posicionamento bibliográfico, mas deve ser delimitada com cuidado.

A formulação atual prevê a existência de episódios críticos em um horizonte futuro. Ela não estima tempo restante até falha terminal de um ativo individual. Portanto, a extensão para RUL deve aparecer como trabalho futuro, dependente de nova formulação do alvo, definição de ativos, modelagem de degradação, censura e métricas temporais específicas.

## 11. Limitações reconhecidas

A inclusão do EWMA causal no NB11 amplia a comparação temporal sem alterar a arquitetura principal do pipeline. Esse baseline atua como referência intermediária entre regras instantâneas simples e modelos sequenciais não triviais, permitindo avaliar se a suavização causal do fail_rate já explica parte do desempenho preditivo. Ainda assim, a análise decisória permanece baseada no escore operacional do modelo supervisionado vencedor, preservando a separação entre baselines comparativos e política decisória final.

Mesmo após a ampliação metodológica, permanecem limitações relevantes. A base utilizada é uma amostra do ambiente Borg e não representa diretamente uma infraestrutura bancária real. Os episódios críticos são definidos por critérios estatísticos sobre taxa de falhas agregada, e não por incidentes de negócio rotulados manualmente. A calibração dos escores deve ser interpretada com cautela caso os artefatos anteriores indiquem que o escore funciona melhor como ranking relativo de risco do que como probabilidade operacional direta.

Além disso, a análise de custo usa razões relativas entre falso positivo e falso negativo, não custos financeiros reais medidos em ambiente produtivo. Essa escolha é metodologicamente aceitável para dissertação, mas deve ser apresentada como parametrização de decisão, não como mensuração econômica definitiva.

## 12. Trabalhos futuros

Como trabalhos futuros, recomenda-se: aplicar a metodologia ao conjunto Borg completo ou a dados reais de infraestrutura; validar custos com especialistas de operação; explorar modelos sequenciais apenas se houver volume e estabilidade suficientes; testar adaptação online em cenários não estacionários; e formular uma extensão específica para PHM/RUL quando houver definição clara de ativo, degradação e evento terminal.

## 13. Conclusão de fechamento

O NB16 consolida a dissertação como uma evolução metodologicamente mais robusta do artigo. O foco final permanece na antecipação de criticidade, na avaliação temporal causal e na decisão sob incerteza. O cenário `{MAIN_SCENARIO}` deve ser mantido como referência principal, enquanto `{STRONG_SENSITIVITY_SCENARIO}` deve ser reportado como sensibilidade forte. A LSTM permanece como experimento complementar, e a conexão com PHM/RUL deve ser apresentada com delimitação explícita.
"""


# -----------------------------------------------------------------------------
# 8. Persistência dos artefatos do NB16
# -----------------------------------------------------------------------------

outputs = {}

def save_csv(df: pd.DataFrame, filename: str) -> Path:
    path = REPORTS_PATH / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8")
    outputs[filename] = str(path)
    return path


def save_md(text: str, filename: str) -> Path:
    path = REPORTS_PATH / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    outputs[filename] = str(path)
    return path


def save_json(obj: Any, filename: str) -> Path:
    path = REPORTS_PATH / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    outputs[filename] = str(path)
    return path


save_csv(inventory_df, "16_artifact_inventory.csv")
save_csv(manifest_df, "16_artifact_manifest_sha256.csv")
save_csv(traceability_df, "16_traceability_matrix.csv")
save_csv(expected_df, "16_expected_artifacts_check.csv")
save_csv(dissertation_figures_df, "16_dissertation_figures.csv")
save_csv(empty_dirs_df, "16_intentional_empty_dirs.csv")
save_csv(key_results_df, "16_key_results_summary.csv")
save_csv(decision_log_df, "16_methodological_decision_log.csv")
save_csv(article_deltas_df, "16_article_dissertation_deltas.csv")
save_csv(limitations_df, "16_limitations_addressing.csv")
save_csv(consistency_df, "16_consistency_checks.csv")
save_md(phm_rul_positioning_md, "16_phm_rul_positioning.md")
save_md(narrative_synthesis_md, "16_narrative_synthesis.md")

# Recalcula hashes dos próprios artefatos NB16 recém-gerados para o resumo final.
nb16_output_manifest = []
for filename, path_str in outputs.items():
    p = Path(path_str)
    h, hs = sha256_file(p)
    nb16_output_manifest.append({
        "filename": filename,
        "path": path_str,
        "size_bytes": p.stat().st_size if p.exists() else None,
        "sha256": h,
        "sha256_status": hs,
    })

summary = {
    "notebook": "NB16",
    "title": "Consolidação, Inventário, Rastreabilidade e Fechamento",
    "execution_ts": EXECUTION_TS,
    "project_root": str(PROJECT_ROOT),
    "reports_path": str(REPORTS_PATH),
    "notebooks_path": str(NOTEBOOKS_PATH) if NOTEBOOKS_PATH else None,
    "central_rule": {
        "does_not_recalculate": True,
        "does_not_change_data": True,
        "does_not_open_new_experiments": True,
        "purpose": "validar, consolidar, documentar e apresentar resultados finais",
    },
    "main_scenario": MAIN_SCENARIO,
    "strong_sensitivity_scenario": STRONG_SENSITIVITY_SCENARIO,
    "methodological_position": {
        "train_m2s": "referência principal da dissertação",
        "train_p95": "sensibilidade forte e competitiva",
        "ewma": EWMA_BASELINE_METHOD,
        "lstm": "experimento complementar defensivo",
        "phm_rul": "posicionamento conceitual; não é estimativa formal de RUL",
    },
    "artifact_counts": {
        "inventory_total": int(len(inventory_df)),
        "manifest_total": int(len(manifest_df)),
        "dissertation_figures_total": int(len(dissertation_figures_df)),
        "dissertation_figures_missing_total": int((dissertation_figures_df["source_exists"] == False).sum()) if not dissertation_figures_df.empty else 0,
        "intentional_empty_dirs_total": int(len(empty_dirs_df)),
        "intentional_empty_dirs_ok_total": int((empty_dirs_df["validated_status"] == "intentional_empty").sum()) if not empty_dirs_df.empty else 0,
        "key_results_total": int(len(key_results_df)),
        "consistency_checks_total": int(len(consistency_df)),
        "warnings_total": int((consistency_df["severity"] == "warning").sum()) if not consistency_df.empty else 0,
        "missing_expected_artifacts_total": int((expected_df["exists"] == False).sum()) if not expected_df.empty else 0,
    },
    "traceability_status_counts": traceability_df["status"].value_counts(dropna=False).to_dict() if not traceability_df.empty else {},
    "consistency_status_counts": consistency_df["status"].value_counts(dropna=False).to_dict() if not consistency_df.empty else {},
    "generated_outputs": nb16_output_manifest,
}

save_json(summary, "16_nb16_summary.json")

print("\n" + "=" * 90)
print("Artefatos gerados pelo NB16")
print("=" * 90)
for filename, path_str in outputs.items():
    print(f"- {filename}: {path_str}")

print("\nResumo final:")
print(json.dumps({
    "inventory_total": summary["artifact_counts"]["inventory_total"],
    "key_results_total": summary["artifact_counts"]["key_results_total"],
    "warnings_total": summary["artifact_counts"]["warnings_total"],
    "missing_expected_artifacts_total": summary["artifact_counts"]["missing_expected_artifacts_total"],
    "traceability_status_counts": summary["traceability_status_counts"],
}, ensure_ascii=False, indent=2))

print("\nConclusão:")
print(f"O NB16 consolidou os artefatos existentes sem recalcular experimentos.")
print(f"Cenário principal preservado: {MAIN_SCENARIO}.")
print(f"Sensibilidade forte preservada: {STRONG_SENSITIVITY_SCENARIO}.")
print("Posicionamento PHM/RUL registrado: antecipação de criticidade, não RUL formal.")
print("=" * 90)


Mounted at /content/drive
NB16 — Consolidação, Inventário, Rastreabilidade e Fechamento
Execução iniciada em: 2026-06-10T23:46:35
PROJECT_ROOT : /content/drive/MyDrive/Mestrado
REPORTS_PATH : /content/drive/MyDrive/Mestrado/04-reports
NOTEBOOKS_PATH: /content/drive/MyDrive/Mestrado/PPCOMP_DM/notebooks
Cenário principal preservado     : W5_K24_H12_P1_TRAIN_M2S
Sensibilidade forte preservada   : W5_K24_H12_P1_TRAIN_P95
Artefatos inventariados: 208
                 layer  nb  n
evidencia_complementar 13a 11
    primeiro_ciclo_dsr  00 26
    primeiro_ciclo_dsr  01  2
    primeiro_ciclo_dsr  02  3
    primeiro_ciclo_dsr  03  3
    primeiro_ciclo_dsr  04  3
    primeiro_ciclo_dsr  05  4
    primeiro_ciclo_dsr  06  3
    primeiro_ciclo_dsr  07 19
    primeiro_ciclo_dsr  08 24
    primeiro_ciclo_dsr  09 16
     segundo_ciclo_dsr  10 17
     segundo_ciclo_dsr  11 23
     segundo_ciclo_dsr  12 11
     segundo_ciclo_dsr  13 11
     segundo_ciclo_dsr  14 16
     segundo_ciclo_dsr  15 16

Matriz de

## 10. Conclusão da Etapa

### 10.1 Execução e papel do NB16

O NB16 foi executado de forma adequada ao seu papel de fechamento do pipeline PPCOMP/DM. A etapa não recalculou modelos, não alterou dados, não redefiniu cenários e não abriu novos experimentos. Sua função foi consolidar, inventariar, validar e documentar os artefatos produzidos ao longo dos notebooks anteriores, preservando a rastreabilidade metodológica da dissertação e oferecendo uma visão integrada do percurso experimental.

A execução confirmou o cenário principal preservado:

`W5_K24_H12_P1_TRAIN_M2S`

Também confirmou a sensibilidade forte preservada:

`W5_K24_H12_P1_TRAIN_P95`

Dessa forma, o NB16 cumpriu sua função de encerramento técnico do ciclo NB00–NB16, com ênfase na consolidação da trajetória metodológica em dois ciclos DSR: o primeiro ciclo, correspondente à linha de base metodológica construída pelos notebooks NB00–NB09, e o segundo ciclo, correspondente à camada dissertativa vigente, composta pelos notebooks NB10–NB16. O NB13a foi mantido como evidência complementar associada à avaliação da suficiência do esforço experimental com LSTM.

A execução também incorporou a decisão recente de manter o inventário em nível leve, sem criação de taxonomias pesadas como `ACTIVE`, `ARCHIVED` ou `INTERMEDIATE`. Essa decisão é adequada ao estado atual do diretório, pois a higiene das figuras já foi realizada no disco, as colisões de nomenclatura foram resolvidas e os resíduos de gerações anteriores deixaram de fazer parte do conjunto ativo.

### 10.2 Escopo de fechamento e ausência de recálculo

O NB16 operou exclusivamente como notebook de consolidação. A etapa não reexecutou baselines, não reestimou limiares, não recalculou métricas, não reprocessou cenários e não alterou artefatos experimentais dos notebooks anteriores. Todos os resultados apresentados pelo NB16 foram extraídos de artefatos já existentes, produzidos nos notebooks NB00–NB15 e NB13a.

Essa característica é importante para a governança da dissertação. O NB16 não deve ser interpretado como etapa experimental adicional, mas como etapa de fechamento, validação e documentação. Seu papel é garantir que os resultados já produzidos estejam rastreáveis, consistentes e disponíveis para a redação da dissertação, para a qualificação e para eventuais auditorias metodológicas posteriores.

A regra central registrada no resumo da execução permanece válida: o NB16 não recalcula, não muda dados e não abre novos experimentos. Ele apenas consolida, valida, documenta e apresenta os resultados finais do pipeline.

### 10.3 Inventário consolidado de artefatos

A execução registrou 208 itens no inventário consolidado. Esse total reflete o estado real e limpo do diretório `04-reports` após a remoção ou movimentação das gerações antigas que anteriormente causavam ambiguidade em NB09, NB10, NB11 e NB13.

A distribuição por camada metodológica ficou coerente com a narrativa de dois ciclos DSR:

* primeiro ciclo DSR: artefatos associados aos notebooks NB00–NB09;
* segundo ciclo DSR: artefatos associados aos notebooks NB10–NB16;
* evidência complementar: artefatos associados ao NB13a.

O primeiro ciclo DSR preserva a história metodológica da pesquisa, incluindo exploração inicial, ingestão, limpeza, agregação temporal, detecção de episódios, engenharia de atributos, rotulagem, modelagem supervisionada inicial e figuras finais associadas ao artigo. O segundo ciclo DSR preserva a camada dissertativa vigente, com diagnóstico metodológico de cenários, limiares train-only, baselines temporais, EWMA causal, análise de sensibilidade, LSTM complementar, análise decisória, figuras finais e rastreabilidade. O NB13a permanece como evidência complementar de robustez, sem ser tratado como novo eixo experimental da dissertação.

O inventário registrou ainda o diretório `figures_nb13a_lstm_tuning` como vazio intencional, reconhecendo que o NB13a não produz figuras por desenho metodológico, mas sim artefatos tabulares, JSON, texto e Parquet.

### 10.4 Manifesto SHA-256

O manifesto SHA-256 registrou 207 artefatos com hash aplicável. A diferença entre os 208 itens do inventário e os 207 registros do manifesto é esperada e metodologicamente correta: o item adicional corresponde ao diretório vazio intencional `figures_nb13a_lstm_tuning`, que deve ser rastreado como evidência de desenho do NB13a, mas não possui hash de arquivo aplicável.

Todos os artefatos gerados pelo NB16 apresentaram status de hash válido. Esse registro fortalece a reprodutibilidade e permite identificar alterações futuras nos arquivos de consolidação. O manifesto atua como mecanismo de controle de integridade dos artefatos finais, sem transformar o NB16 em uma camada pesada de governança documental.

O resultado é adequado ao objetivo desta fase: preservar a rastreabilidade suficiente para defesa acadêmica e auditoria metodológica, evitando esforço excessivo com controle de versões intermediárias que já não fazem parte do diretório ativo.

### 10.5 Matriz de rastreabilidade

A matriz de rastreabilidade contemplou os notebooks NB00 a NB15 e o NB13a, com 17 entradas em status `ok`. Todos os notebooks esperados foram reconhecidos, todos os artefatos obrigatórios foram encontrados e não houve indicação de falha na rastreabilidade.

A matriz reforça que o pipeline completo permanece coberto, desde a fase exploratória inicial até o fechamento visual e decisório. Os notebooks NB00–NB09 permanecem associados ao primeiro ciclo DSR; os notebooks NB10–NB15 permanecem associados ao segundo ciclo DSR; e o NB13a aparece como evidência complementar.

Esse resultado é relevante porque evita uma leitura equivocada de que a limpeza das figuras teria apagado a história científica do projeto. A história permanece preservada no inventário e na matriz de rastreabilidade, mas o conjunto citável da dissertação passa a ser controlado por uma tabela específica de figuras e citabilidade.

### 10.6 Artefatos esperados e checagens de consistência

A execução não identificou ausência de artefatos esperados. O campo `missing_expected_artifacts_total` foi igual a zero, indicando que os arquivos essenciais à sustentação metodológica da dissertação estavam disponíveis no momento da execução.

Foram executadas 33 checagens de consistência, todas concluídas com status `ok`. Essas checagens abrangeram, entre outros elementos, os artefatos de decisão do NB10, o cenário principal promovido ao NB11, o modelo vencedor do NB11, os baselines temporais, o EWMA causal, a sensibilidade do NB12, os artefatos da LSTM no NB13, os artefatos decisórios do NB14, as figuras consolidadas do NB15, o diretório vazio intencional do NB13a e as dimensões dos principais arquivos Parquet.

A ausência de warnings e de falhas indica que o pipeline se encontra tecnicamente fechado para fins de redação e qualificação. Nenhuma inconsistência detectada pelo NB16 exige reexecução de notebooks anteriores.

### 10.7 Distribuição dos artefatos por fase do pipeline

A execução consolidou artefatos distribuídos em três grupos principais.

O primeiro grupo corresponde ao primeiro ciclo DSR, representado pelos notebooks NB00–NB09. Esse grupo preserva a construção original do artefato: exploração inicial da base, ingestão, validação, limpeza, normalização temporal, agregação em janelas, detecção de episódios críticos, engenharia de atributos, rotulagem dos estados operacionais, modelagem inicial e figuras finais associadas ao artigo submetido.

O segundo grupo corresponde ao segundo ciclo DSR, representado pelos notebooks NB10–NB16. Esse grupo registra o refinamento dissertativo do artefato: diagnóstico metodológico de cenários, limiares estimados apenas no treino, promoção de cenários ao NB11, comparação com baselines, EWMA causal, seleção do modelo principal, análise de sensibilidade, comparação com LSTM, função de custo, lead time, antecipabilidade, classes de ação e fechamento rastreável.

O terceiro grupo corresponde à evidência complementar do NB13a. Esse grupo não redefine a decisão principal da dissertação, mas fortalece a defesa metodológica ao mostrar que a LSTM não deixou de ser promovida por ausência de esforço mínimo de ajuste. O NB13a responde a uma pergunta provável de banca: se a Regressão Logística venceu porque a LSTM foi mal configurada. A resposta, preservada nos artefatos do NB13a, é que o tuning compacto não superou a conclusão do NB13 nem atingiu o gatilho de promoção.

### 10.8 Decisão metodológica central

A decisão metodológica central foi preservada: o cenário `W5_K24_H12_P1_TRAIN_M2S` permanece como referência principal da dissertação, enquanto o cenário `W5_K24_H12_P1_TRAIN_P95` permanece como sensibilidade forte e competitiva.

Essa distinção é importante porque mantém a governança metodológica definida após o NB12. A troca do cenário principal somente seria reaberta se o cenário `TRAIN_P95` superasse o cenário `TRAIN_M2S` por margem superior a 5 pontos percentuais em F1 médio no TimeSeriesSplit. Como essa condição não foi atendida, a dissertação preserva `TRAIN_M2S` como eixo principal e utiliza `TRAIN_P95` como evidência de robustez.

A margem de 5 pontos percentuais deve ser interpretada como critério de relevância prática e estabilidade metodológica, não como teste estatístico formal. Essa interpretação deve ser preservada no texto da dissertação para evitar sobredeclaração inferencial.

### 10.9 Consolidação dos resultados-chave

O NB16 consolidou 51 resultados-chave extraídos dos artefatos existentes. Esses resultados sintetizam decisões, métricas, baselines, análises de sensibilidade, posicionamentos metodológicos e registros de rastreabilidade produzidos nos notebooks anteriores.

Entre os pontos consolidados, destacam-se:

* a preservação do cenário principal `W5_K24_H12_P1_TRAIN_M2S`;
* a preservação do cenário `W5_K24_H12_P1_TRAIN_P95` como sensibilidade forte;
* a seleção da Regressão Logística como modelo principal no NB11;
* o desempenho do modelo vencedor, com F1 médio TSCV de aproximadamente 0,5774;
* o recall médio TSCV de aproximadamente 0,8331;
* a precisão média TSCV de aproximadamente 0,4819;
* o ROC-AUC médio de aproximadamente 0,6474;
* o Brier Score de aproximadamente 0,2514;
* a presença dos artefatos de métricas, baselines, lead time, calibração e antecipação de episódios;
* o registro do EWMA causal como baseline temporal suavizado no NB11;
* a preservação da análise de sensibilidade do NB12;
* a preservação da LSTM como experimento complementar;
* a consolidação da função de custo e dos limiares operacionais no NB14;
* a incorporação das figuras finais do NB15 como artefatos de comunicação científica.

Esses resultados não substituem as análises detalhadas dos notebooks anteriores, mas funcionam como índice consolidado para apoiar a redação dos capítulos de resultados, discussão, limitações e conclusão da dissertação.

### 10.10 Baselines temporais e EWMA causal

O NB16 registrou explicitamente a presença do EWMA causal nos artefatos do NB11. Esse registro é metodologicamente importante porque responde a uma limitação identificada após a versão do artigo: a ausência de baseline temporal simples, competitivo e interpretável.

O EWMA causal ocupa papel intermediário na narrativa experimental. Ele não substitui o modelo supervisionado principal, não redefine os escores usados na análise decisória do NB14 e não altera a seleção da Regressão Logística. Seu papel é servir como referência temporal suavizada entre regras simples e modelos supervisionados mais ricos.

A dissertação deve preservar essa distinção. O EWMA causal fortalece a comparação metodológica, mas o escore operacional utilizado para decisão sob incerteza permanece vinculado ao modelo vencedor selecionado no fluxo NB11–NB14.

A consistência EWMA entre NB11 e NB13 já estava garantida por artefato antes desta execução, e o NB16 consolidou a existência dessa cadeia sem exigir reexecução do NB13.

### 10.11 LSTM e NB13a como evidência complementar

A execução preservou a LSTM como experimento complementar, não como modelo principal. Essa decisão é coerente com os resultados consolidados: a LSTM apresentou desempenho próximo, mas inferior ao modelo tabular vencedor, e não atingiu o critério de promoção previamente definido.

O NB13a foi corretamente tratado como evidência complementar. Sua função é defensiva e científica: verificar se a conclusão desfavorável à promoção da LSTM poderia decorrer de subajuste de hiperparâmetros. Os artefatos do NB13a indicam que o tuning compacto não inverteu a conclusão e não justificou substituir os escores do NB11 como insumo da análise decisória do NB14.

Essa leitura evita duas interpretações inadequadas. A primeira seria ignorar redes recorrentes, o que poderia fragilizar a defesa diante de uma banca interessada em LSTM. A segunda seria promover a LSTM sem ganho suficiente, apenas por apelo metodológico. A solução adotada é equilibrada: a LSTM permanece como comparação complementar válida, mas a Regressão Logística permanece como modelo principal por desempenho, parcimônia e estabilidade relativa no protocolo adotado.

### 10.12 Registro do diretório vazio intencional do NB13a

O NB16 registrou o diretório `figures_nb13a_lstm_tuning` como vazio intencional. Esse registro corrige uma ambiguidade que poderia surgir na leitura do acervo: a ausência de figuras no NB13a não indica falha de execução, perda de arquivo ou incompletude do notebook.

O NB13a produz evidência tabular, textual e serializada, incluindo resultados de tuning, agregações por configuração, melhor configuração, resultados em corte fixo, escores, resumo e conclusão. O diretório de figuras vazio é coerente com esse desenho metodológico.

Com isso, o NB16 transforma uma possível dúvida operacional em decisão documentada. Esse ponto deve ser preservado no parecer de figuras e no mapa de escrita, para que a banca não interprete a pasta vazia como lacuna experimental.

### 10.13 Figuras da dissertação e citabilidade

A execução gerou o artefato `16_dissertation_figures.csv`, que mapeia as figuras previstas para a dissertação, seus arquivos de origem, notebooks geradores, ciclo DSR, citabilidade e status de existência. Esse artefato deve passar a ser a fonte preferencial do plano de figuras do mapa de escrita.

Foram registradas 19 entradas. Dessas, 16 correspondem a arquivos já existentes e 3 correspondem a figuras de síntese ainda pendentes, a serem produzidas fora dos notebooks experimentais:

* `framework_completo.pdf`;
* `ciclo_dsr.pdf`;
* `evolucao_metodologica.pdf`.

Essas três entradas ausentes não representam falha do NB16. Elas pertencem à camada de redação e comunicação da dissertação, não ao pipeline experimental. Devem ser produzidas como artefatos da dissertação, preferencialmente em diretório próprio, e depois incorporadas ao plano de figuras.

O NB16 também consolidou a decisão de citabilidade. As figuras do segundo ciclo DSR e as figuras finais do NB15 são as principais candidatas ao corpo do texto. Figuras do primeiro ciclo DSR podem ser usadas de forma indireta ou em contexto de evolução metodológica, mas não devem competir com as figuras canônicas do segundo ciclo na apresentação dos resultados finais.

### 10.14 Figura central do NB10 e remoção da ambiguidade de pastas

A execução passa a operar sobre o estado atual do diretório, no qual a pasta antiga `figures_nb10_thresholds` não integra mais o conjunto ativo. O NB10 vigente está associado à pasta `figures_nb10_scenarios`, que contém as figuras canônicas do diagnóstico metodológico de cenários.

A figura `fig_03_w5_fail_rate_time_thresholds.png`, localizada em `figures_nb10_scenarios`, deve ser tratada como figura central do segundo ciclo DSR. Ela evidencia visualmente a transição metodológica entre o limiar global retrospectivo e a formulação train-only, funcionando como uma das peças visuais mais importantes para explicar por que o segundo ciclo dissertativo foi necessário.

Essa decisão elimina a ambiguidade anterior entre figuras geradas em momentos diferentes do NB10. A dissertação deve usar exclusivamente as figuras vigentes de `figures_nb10_scenarios` como fonte citável do NB10.

### 10.15 Deltas entre artigo e dissertação

O NB16 consolidou os principais deltas entre a linha de base do artigo e a versão dissertativa ampliada. Essa consolidação é relevante porque permite narrar a evolução metodológica sem depender da aceitação ou rejeição do artigo submetido.

A linha de base do artigo, representada pelos notebooks NB00–NB09, estabeleceu a viabilidade da formulação, a construção da série temporal, a definição de episódios e estados operacionais, a modelagem supervisionada inicial e as figuras principais do artigo. O segundo ciclo DSR, representado pelos notebooks NB10–NB16, aprofunda essa base e endereça limitações relevantes.

Entre os principais deltas consolidados estão:

* substituição do limiar global como eixo principal por limiares train-only;
* manutenção do limiar global apenas como referência histórica retrospectiva;
* inclusão de baselines temporais simples e EWMA causal;
* análise de sensibilidade com preservação do cenário principal;
* avaliação complementar com LSTM;
* análise paramétrica de custo por limiar `tau`;
* consolidação de lead time, antecipabilidade e classes de ação;
* geração de figuras finais mais adequadas à dissertação;
* delimitação conceitual frente à literatura de PHM/RUL.

A dissertação, portanto, não invalida a formulação inicial. Ela a amplia, corrige riscos metodológicos, torna a narrativa mais defensável e fortalece a contribuição como framework preditivo e reprodutível para decisão sob incerteza.

### 10.16 Limitações endereçadas

O NB16 consolidou quatro limitações principais originalmente associadas à versão do artigo e registrou como elas foram endereçadas ao longo da versão dissertativa ampliada.

A ausência de baseline temporal robusto foi endereçada no NB11 por meio da comparação com baselines temporais simples, EWMA causal e modelos supervisionados sob protocolo temporal.

O risco de vazamento temporal por limiar global foi endereçado no NB10 pela introdução de cenários com limiares estimados apenas no trecho de treino, preservando o cenário global apenas como referência retrospectiva do primeiro ciclo DSR.

A formulação ainda incompleta da função `C(tau)` foi endereçada no NB14, com análise paramétrica de custo, identificação de limiares ótimos por razão `cFP:cFN`, avaliação de falsos positivos, falsos negativos, falsos alertas, lead time e antecipabilidade.

A ausência de conexão explícita com PHM/RUL foi endereçada no próprio NB16 por delimitação conceitual. O trabalho foi posicionado como antecipação de criticidade operacional e apoio à decisão sob incerteza, sem reivindicar estimativa formal de vida útil remanescente.

Essas limitações permanecem importantes para a discussão, mas deixam de ser lacunas não tratadas. Elas passam a ser pontos reconhecidos, endereçados e delimitados por artefatos específicos do pipeline.

### 10.17 Posicionamento PHM/RUL

O NB16 registrou o posicionamento conceitual do trabalho em relação à literatura de PHM e RUL. A dissertação deve ser apresentada como uma abordagem de antecipação de criticidade operacional e apoio à decisão sob incerteza, não como estimativa formal de *Remaining Useful Life*.

Essa delimitação evita sobredeclaração da contribuição. A formulação atual prevê a ocorrência de janelas críticas futuras dentro de um horizonte operacional `H`. Ela não estima tempo restante até falha terminal de um ativo específico, não define curvas de degradação de componentes, não trata censura, não modela vida útil remanescente e não utiliza métricas próprias de erro temporal de RUL.

A relação com PHM é conceitual e metodológica. O trabalho compartilha com PHM a preocupação com antecipação, prevenção operacional, uso de séries temporais e suporte à decisão. Entretanto, sua unidade de análise são janelas, episódios e estados operacionais agregados, não ativos físicos ou lógicos individualizados com trajetória de degradação terminal.

A extensão para RUL deve ser apresentada como trabalho futuro, dependente de nova formulação do alvo, definição de ativo, definição de evento terminal, tratamento de censura e escolha de métricas temporais específicas.

### 10.18 Síntese narrativa e apoio à redação

O NB16 produziu o arquivo `16_narrative_synthesis.md`, que organiza a trajetória metodológica do projeto em formato narrativo. Esse artefato deve ser utilizado como apoio direto à redação da dissertação, especialmente nas seções de metodologia, resultados, discussão, limitações, contribuições e trabalhos futuros.

A síntese narrativa preserva a diferença entre a linha de base do artigo e a versão dissertativa ampliada. Também registra que os notebooks NB10–NB15 não invalidam a versão submetida ao SBPO, mas aprofundam o pipeline para fins de dissertação.

Esse ponto é essencial para a escrita autocontida. A dissertação não precisa depender da submissão, do parecer ou da aprovação do artigo. A evolução deve ser narrada como maturação interna do artefato em ciclos DSR: primeiro ciclo de formulação e viabilidade; segundo ciclo de robustez, causalidade temporal, baselines, sensibilidade, decisão sob incerteza e rastreabilidade.

### 10.19 Artefatos gerados pelo NB16

O NB16 produziu os seguintes artefatos finais de consolidação:

* `16_artifact_inventory.csv`;
* `16_artifact_manifest_sha256.csv`;
* `16_traceability_matrix.csv`;
* `16_expected_artifacts_check.csv`;
* `16_dissertation_figures.csv`;
* `16_intentional_empty_dirs.csv`;
* `16_key_results_summary.csv`;
* `16_methodological_decision_log.csv`;
* `16_article_dissertation_deltas.csv`;
* `16_limitations_addressing.csv`;
* `16_consistency_checks.csv`;
* `16_phm_rul_positioning.md`;
* `16_narrative_synthesis.md`;
* `16_nb16_summary.json`.

Esses artefatos fecham o ciclo de inventário, manifesto, rastreabilidade, checagem, síntese narrativa, plano de figuras e posicionamento conceitual. Eles devem ser preservados como evidências de apoio à dissertação e como base documental para revisão metodológica futura.

A inclusão de `16_dissertation_figures.csv` e `16_intentional_empty_dirs.csv` representa a principal ampliação da execução atual em relação ao fechamento anterior. Esses dois artefatos registram, respectivamente, o plano de figuras da dissertação e o tratamento explícito do diretório vazio intencional do NB13a.

### 10.20 Pendências remanescentes

A execução do NB16 não deixou pendências experimentais. Não há necessidade de novos modelos, novos folds, novos cenários, novo tuning, nova execução da LSTM ou reexecução do NB13.

As pendências remanescentes são editoriais e externas ao pipeline experimental. A principal delas é a criação das três figuras de síntese da dissertação:

* framework completo;
* ciclo DSR;
* evolução metodológica.

Essas figuras deverão ser produzidas como artefatos da dissertação, fora dos notebooks experimentais. Depois de criadas, poderão ser registradas no plano de figuras e, se desejado, incorporadas a uma versão posterior do inventário final.

Também permanece como tarefa de redação a incorporação da ressalva sobre episódios avaliáveis na discussão de lead time e antecipabilidade, bem como a manutenção da calibração como diagnóstico, não como contribuição central.

Outra tarefa editorial é atualizar o parecer de figuras e o mapa de escrita para refletir o estado atual: diretório limpo, `figures_nb10_thresholds` fora do conjunto ativo, `figures_nb10_scenarios` como única fonte citável do NB10, NB13a com diretório vazio intencional, e `16_dissertation_figures.csv` como fonte preferencial do plano de figuras.

### 10.21 Interpretação para a dissertação

Para a redação da dissertação, a execução do NB16 deve ser usada como evidência de fechamento metodológico, não como capítulo adicional de resultados. O texto deve aproveitar suas consolidações de modo seletivo.

No Capítulo 3, o NB16 apoia a descrição da rastreabilidade, dos ciclos DSR, do controle de artefatos e da governança metodológica. No Capítulo 4, apoia a consistência dos resultados-chave, a seleção do cenário principal, a presença dos baselines e a organização das figuras. No Capítulo 5, apoia a discussão dos deltas entre primeiro e segundo ciclo, as limitações endereçadas, a função do NB13a e o posicionamento PHM/RUL. No Capítulo 6, apoia a síntese final das contribuições, limitações e trabalhos futuros.

O NB16 não deve ser apresentado como fonte de novos achados experimentais. Seu valor está em consolidar os achados e mostrar que eles são rastreáveis, coerentes e metodologicamente fechados.

### 10.22 Avaliação final da etapa

A execução do NB16 foi satisfatória e adequada ao estado atual do projeto. O notebook cumpriu seu papel de fechamento metodológico sem ampliar indevidamente a governança de artefatos e sem introduzir recálculos desnecessários.

A camada leve de classificação por ciclo DSR, citabilidade e mapeamento figura–arquivo foi suficiente para apoiar a redação da dissertação, sem transformar o inventário em uma estrutura pesada. O registro do diretório vazio intencional do NB13a eliminou uma ambiguidade potencial. A tabela de figuras da dissertação criou uma ponte direta entre o pipeline e o mapa de escrita. O posicionamento PHM/RUL delimitou adequadamente o escopo da contribuição.

O pipeline encontra-se metodologicamente fechado para fins de redação e qualificação. O cenário `W5_K24_H12_P1_TRAIN_M2S` permanece como referência principal; `W5_K24_H12_P1_TRAIN_P95` permanece como sensibilidade forte; a Regressão Logística permanece como modelo tabular vencedor; o EWMA causal permanece como baseline temporal suavizado; a LSTM e o NB13a permanecem como evidências complementares; e a análise decisória permanece vinculada ao NB14 e às figuras finais consolidadas no NB15.

Com isso, o NB16 encerra o pipeline experimental NB00–NB16 e transfere o foco do trabalho para a redação da dissertação, a elaboração das figuras de síntese e a consolidação final da narrativa metodológica em ciclos DSR.
